# CS781 Phase 7 — Notebook P7_02: Prior Case Retrieval with the Phase-1 models (bi-encoders)

Every Phase-1 model becomes a **bi-encoder**: a case is cut into 512-token chunks (at sentence boundaries), each chunk
is encoded by the model and mean-pooled into a unit vector, and a case is scored against a query by cosine
similarity. The model is adapted with a **contrastive objective** (query ↔ cited case) using in-batch negatives and
BM25 hard negatives. One notebook runs a queue of experiments (one per GPU); each writes dev scores, its score
matrices and two ready submissions (dense only / dense + BM25 hybrid).

| Step | What happens |
|---|---|
| 1 | config: Phase-1 weights, experiments, queues |
| 2 | inputs: P7_01 output (tokens, gold, BM25 scores), Phase-1 checkpoints, earlier runs (KD teacher) |
| 3 | model classes (NB3) + backbone and adaptation methods (P3: full, partial/top-k layers, IA³, LoRMA, …) |
| 4 | chunking, the retriever (masked mean pooling), encoding a whole pool |
| 5 | scoring: **mean** (one vector per case) and **MaxSim** (each query chunk's best-matching candidate chunk, averaged); Micro-F1@7; hybrid with BM25 |
| 6 | training: InfoNCE with in-batch + hard negatives, **gradient caching** (big batches of long documents on a T4); **KD** from a teacher's score matrix |
| 7 | experiment runner → dev scores, test submissions | 8 | unit tests | 9 | queue | 10 | results |

**Kaggle settings**: GPU **T4 x2**, Internet not needed. **Inputs**: (1) **P7_01 output**, (2) the **Phase-1 weight
datasets**, (3) for a KD student in a later session: the run that trained the teacher.

In [1]:
# ============================================================================================
# SHARED SETUP  —  paste this SAME cell at the top of every Phase-1 notebook.
# Works on Kaggle (Save & Commit or interactive), Colab (Drive) and local machines.
# ============================================================================================
import os, sys, json, time, glob, shutil, random, math, re, zlib, hashlib, gc
from pathlib import Path
from collections import Counter, defaultdict
import numpy as np

os.environ.setdefault("TOKENIZERS_PARALLELISM", "true")

SESSION_START = time.time()
TIME_BUDGET_H = 11.0   # Kaggle kills a session at 12h and then saves NOTHING -> every long loop stops itself before this

IN_COLAB  = "google.colab" in sys.modules
IN_KAGGLE = os.path.exists("/kaggle/working")
PROJECT   = "cs781_p1"

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    ROOT, SCRATCH = Path("/content/drive/MyDrive") / PROJECT, Path("/content/scratch")   # Drive = persistent
elif IN_KAGGLE:
    ROOT, SCRATCH = Path("/kaggle/working") / PROJECT, Path("/tmp/cs781_scratch")       # working = saved on commit
else:
    ROOT, SCRATCH = Path("./") / PROJECT, Path("./scratch")
ROOT    = Path(os.environ.get("CS781_ROOT", ROOT))
SCRATCH = Path(os.environ.get("CS781_SCRATCH", SCRATCH))
ROOT.mkdir(parents=True, exist_ok=True); SCRATCH.mkdir(parents=True, exist_ok=True)

ENV = "colab" if IN_COLAB else "kaggle" if IN_KAGGLE else "local"
print(f"env={ENV}  ROOT={ROOT}  SCRATCH={SCRATCH}")
for p in {ROOT, SCRATCH}:
    du = shutil.disk_usage(p); print(f"  disk @ {p}: free {du.free/1e9:.1f} GB / total {du.total/1e9:.1f} GB")


# ---------------------------------------------------------------- small utilities
def time_left_h():
    return TIME_BUDGET_H - (time.time() - SESSION_START) / 3600

def save_json(obj, path):
    """Atomic write: a crash mid-write can never leave a half-written (corrupt) state file."""
    path = Path(path); path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + ".tmp")
    with open(tmp, "w", encoding="utf-8") as f:
        json.dump(obj, f, ensure_ascii=False, indent=1)
    os.replace(tmp, path)

def load_json(path, default=None):
    path = Path(path)
    if not path.exists(): return default
    with open(path, encoding="utf-8") as f: return json.load(f)

def restore_from_kaggle_inputs(subdirs):
    """Kaggle 'Save & Commit' starts every run with an EMPTY /kaggle/working.
    To resume, attach the output of the previous version (Add Input -> Your Work -> Notebook output).
    This copies <PROJECT>/<subdir> from any attached input into ROOT (never overwriting newer local files)."""
    if not IN_KAGGLE: return
    for sub in subdirs:
        for cand in sorted(glob.glob(f"/kaggle/input/**/{PROJECT}/{sub}", recursive=True)):
            n = 0
            for src in Path(cand).rglob("*"):
                if src.is_file():
                    dst = ROOT / sub / src.relative_to(cand)
                    if not dst.exists():
                        dst.parent.mkdir(parents=True, exist_ok=True); shutil.copy2(src, dst); n += 1
            print(f"restored {n} files: {cand} -> {ROOT/sub}")

def find_artifact(rel):
    """Look for ROOT/rel first, then inside any attached Kaggle input."""
    p = ROOT / rel
    if p.exists(): return p
    if IN_KAGGLE:
        hits = sorted(glob.glob(f"/kaggle/input/**/{PROJECT}/{rel}", recursive=True))
        if hits: return Path(hits[0])
    return None

def get_hf_token():
    if os.environ.get("HF_TOKEN"): return os.environ["HF_TOKEN"]
    try:
        if IN_KAGGLE:
            from kaggle_secrets import UserSecretsClient
            return UserSecretsClient().get_secret("HF")
        if IN_COLAB:
            from google.colab import userdata
            return userdata.get("HF")
    except Exception as e:
        print("Could not read secret 'HF':", e)
    return None

HF_TOKEN = get_hf_token()
if HF_TOKEN: os.environ["HF_TOKEN"] = HF_TOKEN
print("HF token found:", bool(HF_TOKEN))

# Optional: a PRIVATE HF dataset repo used only to move YOUR OWN artifacts between accounts/sessions.
HF_ARTIFACT_REPO = None   # e.g. "your-hf-username/cs781-p1-artifacts"

def hf_push(subdir):
    if not HF_ARTIFACT_REPO: print("HF_ARTIFACT_REPO not set; skip push"); return
    from huggingface_hub import HfApi
    api = HfApi(token=HF_TOKEN)
    api.create_repo(HF_ARTIFACT_REPO, repo_type="dataset", private=True, exist_ok=True)
    api.upload_folder(folder_path=str(ROOT / subdir), path_in_repo=subdir,
                      repo_id=HF_ARTIFACT_REPO, repo_type="dataset")
    print("pushed", subdir)

def hf_pull(subdir):
    if not HF_ARTIFACT_REPO: return
    from huggingface_hub import snapshot_download
    snapshot_download(HF_ARTIFACT_REPO, repo_type="dataset", allow_patterns=[f"{subdir}/*", f"{subdir}/**"],
                      local_dir=str(ROOT), token=HF_TOKEN)
    print("pulled", subdir)


# ---------------------------------------------------------------- data utilities
# These MUST be identical in every notebook (the train/val split and cleaning depend on them).
HF_DATASET_REPO = "Exploration-Lab/CS-781-Capstone"
SOURCE_NAMES = ["acts", "high_courts", "supreme_court", "tribunals"]   # processing order (matters for dedup)
VAL_PER_10K  = 100            # 1% of documents of every source are held out (document-level split)
PROMPT_WORDS, CONT_WORDS = 512, 128

def extract_text(rec):
    if rec is None: return ""
    if isinstance(rec, str): return rec
    if isinstance(rec, list): return "\n".join(extract_text(r) for r in rec)
    if isinstance(rec, dict):
        for k in ("text", "content", "document", "judgment", "judgement", "body"):
            if k in rec: return extract_text(rec[k])
    return ""

def iter_docs(path):
    """Stream documents from a (possibly multi-GB) JSON file WITHOUT loading it all into RAM.
    Handles: [ {...}, ... ]   |   { id: {...}, ... }   |   JSON-lines."""
    import ijson
    with open(path, "rb") as f:
        head = f.read(1 << 16).lstrip()
        f.seek(0)
        if head[:1] == b"[":
            for rec in ijson.items(f, "item", use_float=True):
                yield extract_text(rec)
            return
        first_line = f.readline(50_000_000); second = f.read(1 << 10).lstrip()[:1]
        f.seek(0)
        is_jsonl = False
        if first_line.endswith(b"\n") and second == b"{":
            try: json.loads(first_line); is_jsonl = True
            except Exception: pass
        if is_jsonl:
            for line in f:
                if line.strip(): yield extract_text(json.loads(line))
        else:
            for _, rec in ijson.kvitems(f, "", use_float=True):
                yield extract_text(rec)

_ZW = re.compile(r"[​‌‍﻿­]")
_WS_ALL = re.compile(r"\s+")
_HSPACE = re.compile(r"[^\S\n]+")
_MANY_NL = re.compile(r"\n\s*\n\s*\n+")

def normalize(text, mode):
    """'collapse_all'  -> every whitespace run becomes one space (text looks like ' '.join(text.split()))
       'keep_newlines' -> collapse spaces/tabs, keep at most one blank line"""
    text = _ZW.sub("", text)
    if mode == "collapse_all":
        return _WS_ALL.sub(" ", text).strip()
    text = _HSPACE.sub(" ", text.replace("\r\n", "\n").replace("\r", "\n"))
    text = "\n".join(l.strip() for l in text.split("\n"))
    return _MANY_NL.sub("\n\n", text).strip()

def hash64(text):
    return int.from_bytes(hashlib.blake2b(text.encode("utf-8", "ignore"), digest_size=8).digest(), "little")

def doc_split(source, idx):
    return "val" if zlib.crc32(f"{source}:{idx}".encode()) % 10000 < VAL_PER_10K else "train"

def list_phase1_files(local_raw_dir=None):
    """Return {'train': {source: repo_path}, 'test': repo_path, 'sizes': {repo_path: bytes}}."""
    if local_raw_dir:
        files = [str(Path(p).relative_to(local_raw_dir)) for p in glob.glob(f"{local_raw_dir}/**/*.json*", recursive=True)]
        sizes = {f: os.path.getsize(Path(local_raw_dir) / f) for f in files}
    else:
        from huggingface_hub import HfApi
        api = HfApi(token=HF_TOKEN)
        try:
            entries = list(api.list_repo_tree(HF_DATASET_REPO, repo_type="dataset",
                                              path_in_repo="Phase_1_Pretraining", recursive=True))
            files = [e.path for e in entries if getattr(e, "size", None) is not None]
            sizes = {e.path: e.size for e in entries if getattr(e, "size", None) is not None}
        except Exception as e:
            print("list_repo_tree failed (", e, ") -> falling back to list_repo_files")
            files = [f for f in api.list_repo_files(HF_DATASET_REPO, repo_type="dataset") if f.startswith("Phase_1")]
            sizes = {}
    out = {"train": {}, "test": None, "sizes": sizes}
    for f in files:
        name = Path(f).name.lower()
        if not name.endswith((".json", ".jsonl")): continue
        if "test" in name: out["test"] = f
        for s in SOURCE_NAMES:
            if name.startswith("train") and s in name: out["train"][s] = f
    return out

def get_raw_file(repo_path, local_raw_dir=None):
    if local_raw_dir:
        return Path(local_raw_dir) / repo_path
    from huggingface_hub import hf_hub_download
    return Path(hf_hub_download(HF_DATASET_REPO, repo_path, repo_type="dataset", token=HF_TOKEN,
                                local_dir=str(SCRATCH / "raw")))

def drop_raw_file(path, local_raw_dir=None):
    if local_raw_dir: return          # never delete user-provided files
    try: os.remove(path); print("deleted raw", path)
    except OSError: pass

env=kaggle  ROOT=/kaggle/working/cs781_p1  SCRATCH=/tmp/cs781_scratch
  disk @ /kaggle/working/cs781_p1: free 20.9 GB / total 21.0 GB
  disk @ /tmp/cs781_scratch: free 1038.2 GB / total 8758.4 GB
HF token found: True


## 1. Configuration
* `attn`: E (prefix-LM) and B (encoder) read text bidirectionally, as in pretraining; A0/A1/F are causal (`"bidir"`
  can be forced to test it).
* `method`: `frozen` (no training: how well do Phase-1 representations transfer?), `full`, `topk` (partial /
  layer-wise fine-tuning of the last `topk` blocks), `ia3`, `lorma` (LoRMA⁺, Bihany et al. 2025).
* Training sees `chunks_train` random chunks per case (mean-pooled); evaluation uses up to `max_chunks` chunks
  spread evenly over the case.
* KD: `teacher` = a finished experiment whose `train_scores.npy` (dense scores train queries × train pool) is
  distilled into the student: `α·InfoNCE + (1−α)·KL(p_teacher ‖ p_student)` over the batch's candidates.

In [2]:
import torch, torch.nn as nn, torch.nn.functional as F, zipfile, threading, traceback
import pandas as pd
pd.set_option("display.width", 250); pd.set_option("display.max_columns", 30)
try: display
except NameError: display = print

P7_SMOKE = os.environ.get("CS781_P7_SMOKE") == "1"
P1_SOURCES = {
    "A0": dict(dataset="cs781-p1-model-a0-sc80", run_name="A0_gpt2_110M_SC80_epoch_1000M"),
    "A1": dict(dataset="cs781-p1-model-a1-sc80", run_name="A1_modern_110M_SC80_epoch_1000M"),
    "B":  dict(dataset="cs781-p1-model-b", run_name="B_encdec_6x6_SC80_epoch_1000M"),
    "E":  dict(dataset="data-partial-e-prefixlm-110m-sc80-epoch-1000m", run_name="E_prefixlm_110M_SC80_epoch_1000M"),
    "F":  dict(dataset="cs781-p1-model-f", run_name="F_localglobal_gqa_alibi_SC80_epoch_1000M"),
}
DEFAULTS = dict(
    model="E", attn=None, keep_layers=None, method="lorma", lr=None, topk=2,
    lora_r=16, lora_alpha=32, lora_targets="all", lorma_variant="plus",
    chunk_tok=512, chunks_train=3, max_chunks=16, epochs=4, batch=8, micro=8, hard_negs=1, hard_from=30,
    tau=0.05, weight_decay=0.01, warmup=0.1, grad_clip=1.0,
    teacher=None, kd_alpha=0.5, kd_T=1.0, save_train_scores=False, test=True, seed=0, est_h=None,
    qview="doc", ctx_train=8, ctx_window=1, max_ctx=16, from_exp=None,
)
METHOD_LR = {"full": 5e-5, "topk": 1e-4, "lorma": 5e-4, "ia3": 3e-3, "frozen": 0.0}
EXPERIMENTS = {
    # ---- transfer: Phase-1 representations without any training (dev only, a few minutes each) ----
    "zs_E":  dict(model="E", method="frozen", epochs=0, test=False),
    "zs_B":  dict(model="B", method="frozen", epochs=0, test=False),
    "zs_A1": dict(model="A1", method="frozen", epochs=0, test=False),
    "zs_A1_bidir": dict(model="A1", attn="bidir", method="frozen", epochs=0, test=False),
    "zs_F":  dict(model="F", method="frozen", epochs=0, test=False),
    "zs_A0": dict(model="A0", method="frozen", epochs=0, test=False),
    # ---- account 1: LoRMA on E (also the KD teacher), partial FT on B, the distilled student ----
    "E_lorma":       dict(model="E", method="lorma", save_train_scores=True),
    "B_topk":        dict(model="B", method="topk", topk=2),
    "A0_6L_kd":      dict(model="A0", keep_layers=6, method="full", lr=1e-4, teacher="E_lorma"),
    # ---- account 2: full FT on A1, IA³ on F, the KD control, full FT on E (reference for LoRMA) ----
    "A1_full":       dict(model="A1", method="full"),
    "F_ia3":         dict(model="F", method="ia3"),
    "A0_6L_control": dict(model="A0", keep_layers=6, method="full", lr=1e-4),
    "E_full":        dict(model="E", method="full"),
    # ---- round 2: twice the epochs, 3 BM25 hard negatives per query (attach round-1 acc1 output for the KD teacher) ----
    "E_lorma_r2":       dict(model="E", method="lorma", epochs=8, hard_negs=3),
    "A0_6L_kd_r2":      dict(model="A0", keep_layers=6, method="full", lr=1e-4, epochs=8, hard_negs=3, teacher="E_lorma", kd_alpha=0.3),
    "A1_full_r2":       dict(model="A1", method="full", epochs=8, hard_negs=3),
    "F_ia3_r2":         dict(model="F", method="ia3", lr=1e-2, epochs=8, hard_negs=3),
    "A0_6L_control_r2": dict(model="A0", keep_layers=6, method="full", lr=1e-4, epochs=8, hard_negs=3),
    # ---- round 3 (needs P7_01 v2): lexical++ hybrid, citation-context queries ----
    # *_rs = round-1 model re-scored (no training; attach that account's round-1 P7_02 output)
    "E_lorma_rs":   dict(from_exp="E_lorma"),
    "A0_6L_kd_rs":  dict(from_exp="A0_6L_kd"),
    "A1_full_rs":   dict(from_exp="A1_full"),
    "F_ia3_rs":     dict(from_exp="F_ia3"),
    # *_ctx = trained with citation-context queries (multiple-instance: a case is scored by its best-matching context)
    "E_lorma_ctx":  dict(model="E", method="lorma", qview="ctx", epochs=8, hard_negs=3, save_train_scores=True),
    "A0_6L_kd_ctx": dict(model="A0", keep_layers=6, method="full", lr=1e-4, qview="ctx", epochs=8, hard_negs=3, teacher="E_lorma_ctx", kd_alpha=0.3),
    "A1_full_ctx":  dict(model="A1", method="full", qview="ctx", epochs=8, hard_negs=3),
    "F_ia3_ctx":    dict(model="F", method="ia3", lr=1e-2, qview="ctx", epochs=8, hard_negs=3),
    "A0_6L_control_ctx": dict(model="A0", keep_layers=6, method="full", lr=1e-4, qview="ctx", epochs=8, hard_negs=3),
}
QUEUES = {"acc1": ["E_lorma", "zs_E", "zs_B", "zs_A1", "zs_A1_bidir", "zs_F", "zs_A0", "B_topk", "A0_6L_kd"],
          "acc2": ["A1_full", "F_ia3", "A0_6L_control", "E_full"],
          "acc1_r2": ["E_lorma_r2", "A0_6L_kd_r2"],
          "acc2_r2": ["A1_full_r2", "F_ia3_r2", "A0_6L_control_r2"],
          "acc1_r3": ["E_lorma_ctx", "E_lorma_rs", "A0_6L_kd_rs", "E_lorma_r2", "A0_6L_kd_ctx"],
          "acc2_r3": ["A1_full_ctx", "F_ia3_ctx", "A1_full_rs", "F_ia3_rs", "A0_6L_control_ctx"]}
RUN_QUEUE = QUEUES["acc1_r3"]

if P7_SMOKE:
    P1_SOURCES = {k: dict(dataset="", run_name=f"SMOKE_{k}") for k in P1_SOURCES}
    DEFAULTS.update(epochs=1, max_chunks=4, chunk_tok=128, micro=4)
    for k in EXPERIMENTS:
        if EXPERIMENTS[k].get("keep_layers"): EXPERIMENTS[k]["keep_layers"] = 1
        if EXPERIMENTS[k].get("epochs"): EXPERIMENTS[k]["epochs"] = 1
    RUN_QUEUE = [x for x in os.environ.get("CS781_P7_QUEUE", "E_lorma").split(",") if x]

def exp_cfg(name):
    cfg = dict(DEFAULTS); cfg.update(EXPERIMENTS[name]); cfg["name"] = name
    if cfg["lr"] is None: cfg["lr"] = METHOD_LR[cfg["method"]]
    if cfg["attn"] is None: cfg["attn"] = "bidir" if cfg["model"] in ("E", "B") else "causal"
    if cfg["est_h"] is None: cfg["est_h"] = 0.2 if cfg["method"] == "frozen" or cfg["from_exp"] else (0.3 if cfg["qview"] == "ctx" else 0.25) * cfg["epochs"] + 0.5
    return cfg

EXPS_DIR = ROOT / "p7" / "exps"; EXPS_DIR.mkdir(parents=True, exist_ok=True)
NUM_GPUS = torch.cuda.device_count()
print("GPUs:", NUM_GPUS, "| queue:", RUN_QUEUE)

GPUs: 2 | queue: ['E_lorma_ctx', 'E_lorma_rs', 'A0_6L_kd_rs', 'E_lorma_r2', 'A0_6L_kd_ctx']


## 2. Inputs

In [3]:
META = load_json(find_artifact("p7/data/meta.json")) if find_artifact("p7/data/meta.json") else None
assert META is not None, "attach the P7_01 output (p7/data/…)"
EOS_ID, PAD_ID, BOS_ID = 0, 1, 2

class Docs:
    """token stream + sentence offsets of one split/role (q = queries, c = candidates) from P7_01"""
    def __init__(self, split, role):
        self.tok = np.memmap(find_artifact(f"p7/data/{split}_{role}_tokens.bin"), dtype=np.uint16, mode="r")
        self.so = np.load(find_artifact(f"p7/data/{split}_{role}_sent_off.npy")); self.ds = np.load(find_artifact(f"p7/data/{split}_{role}_doc_sent.npy"))
    def __len__(self): return len(self.ds) - 1
    def chunks(self, d, L):
        """token ranges of document d: whole sentences packed into chunks of ≤ L tokens (a longer sentence is cut)"""
        out, cs, ce = [], None, None
        for i in range(self.ds[d], self.ds[d + 1]):
            a, b = int(self.so[i]), int(self.so[i + 1])
            while b - a > L:
                if cs is not None: out.append((cs, ce)); cs = None
                out.append((a, a + L)); a += L
            if a == b: continue
            if cs is None: cs, ce = a, b
            elif b - cs <= L: ce = b
            else: out.append((cs, ce)); cs, ce = a, b
        if cs is not None: out.append((cs, ce))
        if not out:
            a = int(self.so[self.ds[d]]); out = [(a, a)]
        return out
    def ids(self, r): return np.asarray(self.tok[r[0]:r[1]], dtype=np.int64)
    def ctx_chunks(self, d, sent_ids, L, w=1, cap=None):
        """one window per <CITATION> sentence: the sentence ± w neighbours, cut to ≤ L tokens around the citation sentence"""
        n, base, out = int(self.ds[d + 1] - self.ds[d]), int(self.ds[d]), []
        for i in sent_ids:
            if i >= n: continue
            a, b = int(self.so[base + max(0, i - w)]), int(self.so[base + min(n, i + w + 1)])
            cs, ce = int(self.so[base + i]), int(self.so[base + i + 1])
            if b - a > L: a = max(a, min((cs + ce) // 2 - L // 2, b - L)); b = a + L
            if b > a and (a, b) not in out: out.append((a, b))
        if cap and len(out) > cap: out = [out[j] for j in np.linspace(0, len(out) - 1, cap).round().astype(int)]
        return out

DOCS = {s: {r: Docs(s, r) for r in ("q", "c")} for s in ("train", "dev", "test")}
IDS = {s: load_json(find_artifact(f"p7/data/{s}_ids.json")) for s in ("train", "dev", "test")}
GOLD = {s: load_json(find_artifact(f"p7/data/{s}_gold.json")) for s in ("train", "dev")}
LEX = {s: np.load(find_artifact(f"p7/data/{s}_lex.npz")) for s in ("train", "dev", "test")}
def canon(x):
    s = str(x).strip(); return (s.lstrip("0") or "0") if s.isdigit() else s
CTXS = {s: (load_json(find_artifact(f"p7/data/{s}_q_ctx.json")) if find_artifact(f"p7/data/{s}_q_ctx.json") else None) for s in ("train", "dev", "test")}
HAS_LEXPP = "best_noyear" in LEX["dev"].files
YEAR_GAMMAS = (0.0, 1.0, 3.0, 100.0) if HAS_LEXPP and META.get("year_gamma", 0) > 0 else (0.0,)
print("citation contexts:", "yes" if CTXS["dev"] is not None else "no (P7_01 v1 data)", "| time-rule penalties tried:", YEAR_GAMMAS)
SELF = {}
for s in ("train", "dev", "test"):
    pos = {canon(x): i for i, x in enumerate(IDS[s]["candidates"])}
    SELF[s] = [(qi, pos[canon(q)]) for qi, q in enumerate(IDS[s]["queries"]) if canon(q) in pos]
for s in ("train", "dev", "test"):
    print(f"{s}: {len(DOCS[s]['q'])} queries, {len(DOCS[s]['c'])} candidates" + (f", {sum(map(len, GOLD[s]['gold']))} gold in pool" if s in GOLD else ""))
print("best lexical system from P7_01:", META["best_lexical_system"])

INPUT_ROOTS = [os.environ["CS781_P7_INPUTS"]] if os.environ.get("CS781_P7_INPUTS") else (["/kaggle/input"] if IN_KAGGLE else [])
EXP_DIRS = {}
for r in INPUT_ROOTS:
    for f in sorted(glob.glob(f"{r}/**/result.json", recursive=True) + glob.glob(f"{r}/**/trainable.pt", recursive=True)):
        p = Path(f).parent
        if p.name in EXPERIMENTS and "p7" in p.parts: EXP_DIRS.setdefault(p.name, []).append(p)
def find_exp_file(name, fname):
    p = EXPS_DIR / name / fname
    if p.exists(): return p
    for d in EXP_DIRS.get(name, []):
        if (d / fname).exists(): return d / fname
    return None
for name, dirs in EXP_DIRS.items():                              # restore small result files of attached earlier runs
    for dd in dirs:
        for src in dd.glob("*"):
            if src.is_file() and src.suffix in (".json", ".zip", ".txt", ".npz") and src.name != "error.txt":
                dst = EXPS_DIR / name / src.name
                if not dst.exists(): dst.parent.mkdir(parents=True, exist_ok=True); shutil.copy2(src, dst)
if EXP_DIRS: print("attached earlier experiments:", sorted(EXP_DIRS))

def find_p1_dir(key):
    src = P1_SOURCES[key]
    roots = [os.environ["CS781_P1_RUNS"]] if os.environ.get("CS781_P1_RUNS") else []
    roots += [str(ROOT / "runs")] + (["/kaggle/input"] if IN_KAGGLE else [])
    cands = sorted({c for r in roots for c in glob.glob(f"{r}/**/weights_final.pt", recursive=True)})
    by_run = [c for c in cands if Path(c).parent.name == src["run_name"]]
    by_ds = [c for c in cands if src["dataset"] and src["dataset"].lower() in [p.lower() for p in Path(c).parts]]
    hit = (by_run or by_ds or [None])[0]
    return Path(hit).parent if hit else None
P1_DIRS = {k: find_p1_dir(k) for k in P1_SOURCES}
for k, p in P1_DIRS.items(): print(f"  {k}: {p if p else 'NOT FOUND'}")

citation contexts: yes | time-rule penalties tried: (0.0, 1.0, 3.0, 100.0)
train: 827 queries, 4320 candidates, 5308 gold in pool
dev: 118 queries, 1023 candidates, 973 gold in pool
test: 237 queries, 1727 candidates
best lexical system from P7_01: lexical++
attached earlier experiments: ['A0_6L_control', 'A0_6L_kd', 'A1_full', 'B_topk', 'E_full', 'E_lorma', 'F_ia3', 'zs_A0', 'zs_A1', 'zs_A1_bidir', 'zs_B', 'zs_E', 'zs_F']
  A0: /kaggle/input/datasets/aarinsheth/cs781-p1-model-a0-sc80/cs781_p1/runs/A0_gpt2_110M_SC80_epoch_1000M
  A1: /kaggle/input/datasets/aarinsheth/cs781-p1-model-a1-sc80/cs781_p1/runs/A1_modern_110M_SC80_epoch_1000M
  B: /kaggle/input/datasets/aaravoswal/cs781-p1-model-b/cs781_p1/runs/B_encdec_6x6_SC80_epoch_1000M
  E: /kaggle/input/datasets/chrishitha2403/data-partial-e-prefixlm-110m-sc80-epoch-1000m/cs781_p1/runs/E_prefixlm_110M_SC80_epoch_1000M
  F: /kaggle/input/datasets/chrishitha2403/cs781-p1-model-f/cs781_p1/runs/F_localglobal_gqa_alibi_SC80_epoch_1000M


## 3. Model classes (NB3) and adaptation methods (P3_02)

In [4]:
# (model classes copied unchanged from NB3's model cell)
class CausalSelfAttention(nn.Module):
    def __init__(self, d, h, dropout):
        super().__init__()
        self.h, self.hd = h, d // h
        self.qkv = nn.Linear(d, 3 * d)
        self.proj = nn.Linear(d, d)
        self.proj.RESIDUAL_SCALE = True
        self.dropout = dropout

    def forward(self, x, past=None, mask=None, use_cache=False):
        B, T, C = x.shape
        q, k, v = self.qkv(x).split(C, dim=2)
        q, k, v = [t.view(B, T, self.h, self.hd).transpose(1, 2) for t in (q, k, v)]
        if past is not None:
            k = torch.cat([past[0], k], dim=2); v = torch.cat([past[1], v], dim=2)
        present = (k, v) if use_cache else None
        drop = self.dropout if self.training else 0.0
        if mask is None:
            y = F.scaled_dot_product_attention(q, k, v, is_causal=True, dropout_p=drop)
        else:
            y = F.scaled_dot_product_attention(q, k, v, attn_mask=mask, dropout_p=drop)
        return self.proj(y.transpose(1, 2).reshape(B, T, C)), present

class MLP(nn.Module):
    def __init__(self, d, dropout):
        super().__init__()
        self.fc, self.proj = nn.Linear(d, 4 * d), nn.Linear(4 * d, d)
        self.proj.RESIDUAL_SCALE = True
        self.drop = nn.Dropout(dropout)
    def forward(self, x):
        return self.drop(self.proj(F.gelu(self.fc(x), approximate="tanh")))

class Block(nn.Module):
    def __init__(self, d, h, dropout):
        super().__init__()
        self.ln1, self.ln2 = nn.LayerNorm(d), nn.LayerNorm(d)
        self.attn, self.mlp = CausalSelfAttention(d, h, dropout), MLP(d, dropout)
        self.drop = nn.Dropout(dropout)
    def forward(self, x, past=None, mask=None, use_cache=False):
        a, present = self.attn(self.ln1(x), past, mask, use_cache)
        x = x + self.drop(a)
        x = x + self.mlp(self.ln2(x))
        return x, present

class GPT2LM(nn.Module):
    def __init__(self, vocab, n_layer, n_head, d_model, ctx_len, dropout=0.0, **_):
        super().__init__()
        self.ctx_len = ctx_len
        self.wte, self.wpe = nn.Embedding(vocab, d_model), nn.Embedding(ctx_len, d_model)
        self.drop = nn.Dropout(dropout)
        self.blocks = nn.ModuleList([Block(d_model, n_head, dropout) for _ in range(n_layer)])
        self.ln_f = nn.LayerNorm(d_model)
        self.lm_head = nn.Linear(d_model, vocab, bias=False)
        self.lm_head.weight = self.wte.weight                     # weight tying
        for name, m in self.named_modules():
            if isinstance(m, nn.Linear):
                std = 0.02 / math.sqrt(2 * n_layer) if getattr(m, "RESIDUAL_SCALE", False) else 0.02
                nn.init.normal_(m.weight, 0.0, std)
                if m.bias is not None: nn.init.zeros_(m.bias)
            elif isinstance(m, nn.Embedding):
                nn.init.normal_(m.weight, 0.0, 0.02)

    def forward(self, idx, targets=None, pos=None, mask=None, cache=None, use_cache=False, last_only=False):
        B, T = idx.shape
        if pos is None: pos = torch.arange(T, device=idx.device)[None]
        x = self.drop(self.wte(idx) + self.wpe(pos))
        new_cache = []
        for i, blk in enumerate(self.blocks):
            x, present = blk(x, cache[i] if cache is not None else None, mask, use_cache)
            new_cache.append(present)
        x = self.ln_f(x)
        if last_only: x = x[:, -1:]
        logits = self.lm_head(x)
        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.float().view(-1, logits.size(-1)), targets.reshape(-1))
        return logits, loss, (new_cache if use_cache else None)

# ------------------------------------------------------------------ A1: modern decoder
class RMSNorm(nn.Module):
    """x / sqrt(mean(x²)) · g — no mean subtraction, no bias (cheaper than LayerNorm, equally stable). Computed in fp32."""
    def __init__(self, d, eps=1e-6):
        super().__init__(); self.eps = eps; self.weight = nn.Parameter(torch.ones(d))
    def forward(self, x):
        xf = x.float()
        return (xf * torch.rsqrt(xf.pow(2).mean(-1, keepdim=True) + self.eps) * self.weight).to(x.dtype)

def rope_tables(head_dim, n_pos, base):
    inv = 1.0 / (base ** (torch.arange(0, head_dim, 2).float() / head_dim))
    ang = torch.outer(torch.arange(n_pos).float(), inv)                    # (n_pos, head_dim/2)
    return torch.cos(ang), torch.sin(ang)

def apply_rope(x, cos, sin):
    """Rotate each (x1_i, x2_i) pair of the head dimension by angle pos·θ_i. The dot product q·k then depends only on
    the RELATIVE distance between the two positions. x: (B, H, T, D); cos/sin: (B or 1, 1, T, D/2)."""
    d = x.size(-1) // 2
    x1, x2 = x[..., :d].float(), x[..., d:].float()
    return torch.cat([x1 * cos - x2 * sin, x1 * sin + x2 * cos], -1).to(x.dtype)

class ModernAttention(nn.Module):
    def __init__(self, d, h, kv, dropout, rope, causal=True):
        super().__init__()
        self.h, self.kv, self.hd, self.rope, self.dropout, self.causal = h, kv, d // h, rope, dropout, causal
        self.q = nn.Linear(d, d, bias=False)
        self.kvp = nn.Linear(d, 2 * kv * self.hd, bias=False)             # GQA when kv < h: fewer key/value heads
        self.o = nn.Linear(d, d, bias=False); self.o.RESIDUAL_SCALE = True
    def forward(self, x, cos, sin, past=None, mask=None, use_cache=False):
        B, T, C = x.shape
        q = self.q(x).view(B, T, self.h, self.hd).transpose(1, 2)
        k, v = self.kvp(x).view(B, T, 2, self.kv, self.hd).permute(2, 0, 3, 1, 4)
        if self.rope:
            q, k = apply_rope(q, cos, sin), apply_rope(k, cos, sin)
        if past is not None:
            k = torch.cat([past[0], k], 2); v = torch.cat([past[1], v], 2)
        present = (k, v) if use_cache else None
        if self.kv != self.h:
            k, v = k.repeat_interleave(self.h // self.kv, 1), v.repeat_interleave(self.h // self.kv, 1)
        drop = self.dropout if self.training else 0.0
        # no explicit mask: causal for decoders (but a single cached query may see every past key), full for encoders
        y = (F.scaled_dot_product_attention(q, k, v, is_causal=self.causal and past is None, dropout_p=drop)
             if mask is None else F.scaled_dot_product_attention(q, k, v, attn_mask=mask, dropout_p=drop))
        return self.o(y.transpose(1, 2).reshape(B, T, C)), present

class SwiGLU(nn.Module):
    """out = W2( SiLU(W1 x) ⊙ W3 x ): a learned gate decides, per hidden unit, how much of W3 x passes through."""
    def __init__(self, d, hidden, dropout):
        super().__init__()
        self.w1, self.w3 = nn.Linear(d, hidden, bias=False), nn.Linear(d, hidden, bias=False)
        self.w2 = nn.Linear(hidden, d, bias=False); self.w2.RESIDUAL_SCALE = True
        self.drop = nn.Dropout(dropout)
    def forward(self, x):
        return self.drop(self.w2(F.silu(self.w1(x)) * self.w3(x)))

class GeluMLP(nn.Module):
    def __init__(self, d, hidden, dropout):
        super().__init__()
        self.fc, self.proj = nn.Linear(d, hidden, bias=False), nn.Linear(hidden, d, bias=False)
        self.proj.RESIDUAL_SCALE = True; self.drop = nn.Dropout(dropout)
    def forward(self, x):
        return self.drop(self.proj(F.gelu(self.fc(x), approximate="tanh")))

class ModernBlock(nn.Module):
    def __init__(self, d, h, kv, hidden, dropout, norm, mlp, rope):
        super().__init__()
        Norm = RMSNorm if norm == "rms" else nn.LayerNorm
        self.n1, self.n2 = Norm(d), Norm(d)
        self.attn = ModernAttention(d, h, kv, dropout, rope)
        self.mlp = SwiGLU(d, hidden, dropout) if mlp == "swiglu" else GeluMLP(d, hidden, dropout)
        self.drop = nn.Dropout(dropout)
    def forward(self, x, cos, sin, past=None, mask=None, use_cache=False):
        a, present = self.attn(self.n1(x), cos, sin, past, mask, use_cache)
        x = x + self.drop(a)
        return x + self.mlp(self.n2(x)), present

class ModernLM(nn.Module):
    def __init__(self, vocab, n_layer, n_head, d_model, ctx_len, dropout=0.0, n_kv_head=None, ffn_hidden=2048,
                 rope_base=10000, pos="rope", norm="rms", mlp="swiglu", **_):
        super().__init__()
        self.ctx_len, self.use_rope = ctx_len, pos == "rope"
        self.wte = nn.Embedding(vocab, d_model)
        self.wpe = None if self.use_rope else nn.Embedding(ctx_len, d_model)
        hd = d_model // n_head
        cos, sin = rope_tables(hd, 2 * ctx_len, rope_base)
        self.register_buffer("rope_cos", cos, persistent=False); self.register_buffer("rope_sin", sin, persistent=False)
        self.drop = nn.Dropout(dropout)
        self.blocks = nn.ModuleList([ModernBlock(d_model, n_head, n_kv_head or n_head, ffn_hidden, dropout, norm, mlp,
                                                 self.use_rope) for _ in range(n_layer)])
        self.norm_f = RMSNorm(d_model) if norm == "rms" else nn.LayerNorm(d_model)
        self.lm_head = nn.Linear(d_model, vocab, bias=False)
        self.lm_head.weight = self.wte.weight
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.normal_(m.weight, 0.0, 0.02 / math.sqrt(2 * n_layer) if getattr(m, "RESIDUAL_SCALE", False) else 0.02)
            elif isinstance(m, nn.Embedding):
                nn.init.normal_(m.weight, 0.0, 0.02)

    def forward(self, idx, targets=None, pos=None, mask=None, cache=None, use_cache=False, last_only=False):
        B, T = idx.shape
        if pos is None: pos = torch.arange(T, device=idx.device)[None]
        x = self.wte(idx)
        if self.wpe is not None: x = x + self.wpe(pos)
        cos, sin = self.rope_cos[pos][:, None], self.rope_sin[pos][:, None]
        x = self.drop(x)
        new_cache = []
        for i, blk in enumerate(self.blocks):
            x, present = blk(x, cos, sin, cache[i] if cache is not None else None, mask, use_cache)
            new_cache.append(present)
        x = self.norm_f(x)
        if last_only: x = x[:, -1:]
        logits = self.lm_head(x)
        loss = None if targets is None else F.cross_entropy(logits.float().view(-1, logits.size(-1)), targets.reshape(-1))
        return logits, loss, (new_cache if use_cache else None)

# ------------------------------------------------------------------ B: encoder-decoder
class CrossAttention(nn.Module):
    """decoder queries attend to the encoder's output ("memory"); keys/values are computed once per prompt"""
    def __init__(self, d, h):
        super().__init__()
        self.h, self.hd = h, d // h
        self.q = nn.Linear(d, d, bias=False); self.kvp = nn.Linear(d, 2 * d, bias=False)
        self.o = nn.Linear(d, d, bias=False); self.o.RESIDUAL_SCALE = True
    def forward(self, x, mem, mem_valid=None, kv=None):
        B, T, C = x.shape
        q = self.q(x).view(B, T, self.h, self.hd).transpose(1, 2)
        if kv is None:
            S = mem.size(1)
            kv = tuple(self.kvp(mem).view(B, S, 2, self.h, self.hd).permute(2, 0, 3, 1, 4))
        k, v = kv
        mask = None if mem_valid is None else mem_valid[:, None, None, :]
        y = F.scaled_dot_product_attention(q, k, v, attn_mask=mask)
        return self.o(y.transpose(1, 2).reshape(B, T, C)), kv

class EncoderBlock(nn.Module):
    def __init__(self, d, h, hidden, dropout):
        super().__init__()
        self.n1, self.n2 = RMSNorm(d), RMSNorm(d)
        self.attn = ModernAttention(d, h, h, dropout, rope=True, causal=False)     # bidirectional
        self.mlp = SwiGLU(d, hidden, dropout)
    def forward(self, x, cos, sin, mask=None):
        x = x + self.attn(self.n1(x), cos, sin, None, mask, False)[0]
        return x + self.mlp(self.n2(x))

class DecoderBlock(nn.Module):
    def __init__(self, d, h, hidden, dropout):
        super().__init__()
        self.n1, self.n2, self.n3 = RMSNorm(d), RMSNorm(d), RMSNorm(d)
        self.self_attn = ModernAttention(d, h, h, dropout, rope=True, causal=True)
        self.cross = CrossAttention(d, h)
        self.mlp = SwiGLU(d, hidden, dropout)
    def forward(self, x, cos, sin, mem, mem_valid, cache=None, use_cache=False):
        a, self_kv = self.self_attn(self.n1(x), cos, sin, cache["self"] if cache else None, None, use_cache)
        x = x + a
        c, cross_kv = self.cross(self.n2(x), mem, mem_valid, cache["cross"] if cache else None)
        x = x + c
        return x + self.mlp(self.n3(x)), ({"self": self_kv, "cross": cross_kv} if use_cache else None)

class EncDecRunner:
    """generation for the encoder-decoder: encode the prompt once, then decode from <|bos|> one token at a time"""
    def __init__(self, model, rows, pad_id, device, amp):
        self.model, self.device, self.amp = model, device, amp
        B, L = len(rows), max(len(r) for r in rows)
        idx = torch.full((B, L), pad_id, dtype=torch.long); valid = torch.zeros((B, L), dtype=torch.bool)
        for r, ids_ in enumerate(rows):                                        # right padding (bidirectional encoder)
            idx[r, :len(ids_)] = torch.tensor(ids_); valid[r, :len(ids_)] = True
        self.idx, self.mem_valid = idx.to(device), valid.to(device)
    def prefill(self):
        with self.amp:
            self.mem = self.model.encode(self.idx, self.mem_valid)
            bos = torch.full((self.idx.size(0), 1), self.model.bos_id, dtype=torch.long, device=self.device)
            logits, self.cache = self.model.decode(bos, self.mem, self.mem_valid, 0, None, use_cache=True)
        self.t = 0
        return logits[:, -1].float()
    def step(self, nxt):
        self.t += 1
        with self.amp:
            logits, self.cache = self.model.decode(nxt[:, None], self.mem, self.mem_valid, self.t, self.cache, use_cache=True)
        return logits[:, -1].float()
    def reorder(self, idx):
        self.mem, self.mem_valid = self.mem[idx], self.mem_valid[idx]; self.cache = reorder_cache(self.cache, idx)

class EncDecLM(nn.Module):
    """B: Transformer encoder-decoder. Training: each window is split at a random point E — the encoder reads tokens
    [0, E) bidirectionally, the decoder (starting from <|bos|>, cross-attending to the encoder) predicts tokens [E, T].
    At test time the prompt goes to the encoder and the continuation is decoded — exactly the task's shape."""
    is_encdec = True
    def __init__(self, vocab, n_layer, n_head, d_model, ctx_len, dropout=0.0, ffn_hidden=2048, rope_base=10000,
                 enc_layers=None, enc_len_eval=768, bos_id=2, **_):
        super().__init__()
        ne = enc_layers or n_layer // 2; nd = n_layer - ne
        self.bos_id, self.enc_len_eval, self.ctx_len = bos_id, enc_len_eval, ctx_len
        self.wte = nn.Embedding(vocab, d_model)
        cos, sin = rope_tables(d_model // n_head, 2 * ctx_len, rope_base)
        self.register_buffer("rope_cos", cos, persistent=False); self.register_buffer("rope_sin", sin, persistent=False)
        self.enc = nn.ModuleList([EncoderBlock(d_model, n_head, ffn_hidden, dropout) for _ in range(ne)])
        self.dec = nn.ModuleList([DecoderBlock(d_model, n_head, ffn_hidden, dropout) for _ in range(nd)])
        self.enc_norm, self.norm_f = RMSNorm(d_model), RMSNorm(d_model)
        self.lm_head = nn.Linear(d_model, vocab, bias=False); self.lm_head.weight = self.wte.weight
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.normal_(m.weight, 0.0, 0.02 / math.sqrt(2 * n_layer) if getattr(m, "RESIDUAL_SCALE", False) else 0.02)
            elif isinstance(m, nn.Embedding):
                nn.init.normal_(m.weight, 0.0, 0.02)
    def _rope(self, start, T, device):
        p = torch.arange(start, start + T, device=device)[None]
        return self.rope_cos[p][:, None], self.rope_sin[p][:, None]
    def encode(self, idx, valid=None):
        cos, sin = self._rope(0, idx.size(1), idx.device)
        x = self.wte(idx); mask = None if valid is None else valid[:, None, None, :]
        for blk in self.enc: x = blk(x, cos, sin, mask)
        return self.enc_norm(x)
    def decode(self, idx, mem, mem_valid, start, cache=None, use_cache=False):
        cos, sin = self._rope(start, idx.size(1), idx.device)
        x = self.wte(idx); new = []
        for i, blk in enumerate(self.dec):
            x, c = blk(x, cos, sin, mem, mem_valid, cache[i] if cache else None, use_cache); new.append(c)
        return self.lm_head(self.norm_f(x)), (new if use_cache else None)
    def forward(self, idx, targets=None, enc_len=None, **_):
        B, T = idx.shape
        E = int(enc_len or min(self.enc_len_eval, T - 1))
        mem = self.encode(idx[:, :E])
        dec_in = torch.cat([torch.full((B, 1), self.bos_id, dtype=torch.long, device=idx.device), idx[:, E:]], 1)
        logits, _ = self.decode(dec_in, mem, None, 0)
        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.float().view(-1, logits.size(-1)), targets[:, E - 1:].reshape(-1))
        return logits, loss, None
    def make_runner(self, rows, pad_id, device, amp):
        return EncDecRunner(self, rows, pad_id, device, amp)

# ------------------------------------------------------------------ C: retention network (RetNet-style), D: hybrid
class Retention(nn.Module):
    """Multi-scale retention (Sun et al. 2023, "Retentive Network"). Attention without softmax:
         parallel form (training):   O = (Q Kᵀ ⊙ D) V,   D[n,m] = γ^(n−m) for n ≥ m, else 0
         recurrent form (generation): S_n = γ·S_{n−1} + k_nᵀ v_n,   o_n = q_n S_n
       Both give identical outputs. Each head has its own decay γ (short- to long-range memory), so generation needs
       only a fixed-size state per head — no growing KV-cache, O(1) cost per new token.
       Per-head normalisation + a SiLU output gate as in RetNet; RoPE on q/k for relative position."""
    def __init__(self, d, h):
        super().__init__()
        self.h, self.hd = h, d // h
        self.q, self.k = nn.Linear(d, d, bias=False), nn.Linear(d, d, bias=False)
        self.v, self.g = nn.Linear(d, d, bias=False), nn.Linear(d, d, bias=False)
        self.o = nn.Linear(d, d, bias=False); self.o.RESIDUAL_SCALE = True
        self.register_buffer("log_gamma", torch.log(1 - 2.0 ** (-5 - torch.arange(h, dtype=torch.float32))), persistent=False)
    def forward(self, x, cos, sin, state=None, valid=None, use_cache=False):
        B, T, C = x.shape
        q, k, v = [m(x).view(B, T, self.h, self.hd).transpose(1, 2) for m in (self.q, self.k, self.v)]
        q, k = apply_rope(q, cos, sin), apply_rope(k, cos, sin)
        if valid is not None:                                   # left padding: padded positions contribute nothing
            vm = valid[:, None, :, None].to(k.dtype); k, v = k * vm, v * vm
        q = q * self.hd ** -0.5
        lg = self.log_gamma.view(1, self.h, 1, 1)
        new_state = None
        if state is None:                                       # parallel form
            n = torch.arange(T, device=x.device, dtype=torch.float32)
            diff = n[:, None] - n[None, :]
            D = torch.where(diff >= 0, torch.exp(diff * lg), torch.zeros((), device=x.device)).to(q.dtype)
            o = ((q @ k.transpose(-1, -2)) * D) @ v
            if use_cache:
                w = torch.exp((T - 1 - n).view(1, 1, T, 1) * lg)                       # γ^(T−1−m)
                new_state = (k.float() * w).transpose(-1, -2) @ v.float()            # (B, H, hd, hd)
        else:                                                   # recurrent form, one token at a time
            new_state = state * torch.exp(lg) + k.float().transpose(-1, -2) @ v.float()
            o = (q.float() @ new_state).to(q.dtype)
        o = F.layer_norm(o.float(), (self.hd,)).to(x.dtype)     # per-head normalisation (GroupNorm, no affine)
        o = o.transpose(1, 2).reshape(B, T, C)
        return self.o(F.silu(self.g(x)) * o), (new_state if use_cache else None)

class RetBlock(nn.Module):
    def __init__(self, d, h, hidden, dropout):
        super().__init__()
        self.n1, self.n2 = RMSNorm(d), RMSNorm(d)
        self.ret, self.mlp = Retention(d, h), SwiGLU(d, hidden, dropout)
    def forward(self, x, cos, sin, state=None, valid=None, use_cache=False):
        r, s = self.ret(self.n1(x), cos, sin, state, valid, use_cache)
        x = x + r
        return x + self.mlp(self.n2(x)), s

class RetNetLM(nn.Module):
    """C (attn_every=0): every block is retention.  D (attn_every=k): every k-th block is full softmax attention
    (ModernBlock), the rest retention — retention supplies cheap long-range decayed memory, the few attention layers
    give exact recall of specific earlier tokens (names, section numbers)."""
    def __init__(self, vocab, n_layer, n_head, d_model, ctx_len, dropout=0.0, ffn_hidden=1792, rope_base=10000,
                 attn_every=0, n_kv_head=None, **_):
        super().__init__()
        self.ctx_len = ctx_len
        self.wte = nn.Embedding(vocab, d_model)
        cos, sin = rope_tables(d_model // n_head, 2 * ctx_len, rope_base)
        self.register_buffer("rope_cos", cos, persistent=False); self.register_buffer("rope_sin", sin, persistent=False)
        self.kinds = ["attn" if attn_every and (i + 1) % attn_every == 0 else "ret" for i in range(n_layer)]
        self.blocks = nn.ModuleList([
            ModernBlock(d_model, n_head, n_kv_head or n_head, ffn_hidden, dropout, "rms", "swiglu", True) if kd == "attn"
            else RetBlock(d_model, n_head, ffn_hidden, dropout) for kd in self.kinds])
        self.norm_f = RMSNorm(d_model)
        self.lm_head = nn.Linear(d_model, vocab, bias=False); self.lm_head.weight = self.wte.weight
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.normal_(m.weight, 0.0, 0.02 / math.sqrt(2 * n_layer) if getattr(m, "RESIDUAL_SCALE", False) else 0.02)
            elif isinstance(m, nn.Embedding):
                nn.init.normal_(m.weight, 0.0, 0.02)
    def forward(self, idx, targets=None, pos=None, mask=None, cache=None, use_cache=False, last_only=False):
        B, T = idx.shape
        if pos is None: pos = torch.arange(T, device=idx.device)[None]
        cos, sin = self.rope_cos[pos][:, None], self.rope_sin[pos][:, None]
        valid = mask[:, 0, -1, :] if (mask is not None and cache is None) else None   # key validity at prefill
        x = self.wte(idx); new = []
        for i, (kd, blk) in enumerate(zip(self.kinds, self.blocks)):
            c = cache[i] if cache is not None else None
            if kd == "attn": x, c = blk(x, cos, sin, c, mask, use_cache)
            else:            x, c = blk(x, cos, sin, c, valid, use_cache)
            new.append(c)
        x = self.norm_f(x)
        if last_only: x = x[:, -1:]
        logits = self.lm_head(x)
        loss = None if targets is None else F.cross_entropy(logits.float().view(-1, logits.size(-1)), targets.reshape(-1))
        return logits, loss, (new if use_cache else None)

# ------------------------------------------------------------------ E: prefix-LM (encoder/hybrid adapted for LM)
class PrefixLM(ModernLM):
    """E: the A1 stack with a different ATTENTION MASK. Prompt (prefix) tokens attend to the whole prompt in both
    directions, like a BERT encoder; continuation tokens attend causally, like GPT. One shared stack plays both roles.
    Training: split each window at a random E, loss only on the continuation (predictions from position E−1 on).
    Generation: prefill the prompt with the bidirectional mask, then ordinary cached causal decoding."""
    def __init__(self, *a, enc_len_eval=768, **kw):
        super().__init__(*a, **kw); self.enc_len_eval = enc_len_eval
    def forward(self, idx, targets=None, enc_len=None, pos=None, mask=None, cache=None, use_cache=False, last_only=False):
        if targets is None:                                       # generation path (mask supplied by the runner)
            return super().forward(idx, None, pos, mask, cache, use_cache, last_only)
        T = idx.size(1); E = int(enc_len or min(self.enc_len_eval, T - 1))
        i = torch.arange(T, device=idx.device)
        allowed = (i[None, :] <= i[:, None]) | ((i[:, None] < E) & (i[None, :] < E))   # causal ∪ (prefix × prefix)
        logits, _, _ = super().forward(idx, None, None, allowed[None, None], None, False, False)
        loss = F.cross_entropy(logits[:, E - 1:].float().reshape(-1, logits.size(-1)), targets[:, E - 1:].reshape(-1))
        return logits, loss, None
    def make_runner(self, rows, pad_id, device, amp):
        return PrefixRunner(self, rows, pad_id, device, amp)

# ------------------------------------------------------------------ F: efficient local-global decoder
def alibi_slopes(n):
    """ALiBi head slopes (Press et al. 2022): geometric sequence; extra heads interleaved when n is not a power of 2"""
    def pow2(m): return [2 ** (-8 * (k + 1) / m) for k in range(m)]
    if math.log2(n).is_integer(): return pow2(n)
    p = 2 ** math.floor(math.log2(n))
    return pow2(p) + pow2(2 * p)[0::2][: n - p]

class LocalGlobalAttention(nn.Module):
    """Grouped-query attention (h query heads share kv key/value heads) with an ALiBi distance penalty added to the
    attention logits (no position embeddings at all). window=W makes it sliding-window attention: a query only sees
    the previous W tokens; window=None is full causal attention."""
    def __init__(self, d, h, kv, window):
        super().__init__()
        self.h, self.kv, self.hd, self.window = h, kv, d // h, window
        self.q = nn.Linear(d, d, bias=False); self.kvp = nn.Linear(d, 2 * kv * self.hd, bias=False)
        self.o = nn.Linear(d, d, bias=False); self.o.RESIDUAL_SCALE = True
    def forward(self, x, bias, past=None, use_cache=False):
        B, T, C = x.shape
        q = self.q(x).view(B, T, self.h, self.hd).transpose(1, 2)
        k, v = self.kvp(x).view(B, T, 2, self.kv, self.hd).permute(2, 0, 3, 1, 4)
        if past is not None:
            k = torch.cat([past[0], k], 2); v = torch.cat([past[1], v], 2)
        present = (k, v) if use_cache else None
        if self.kv != self.h:
            k, v = k.repeat_interleave(self.h // self.kv, 1), v.repeat_interleave(self.h // self.kv, 1)
        y = F.scaled_dot_product_attention(q, k, v, attn_mask=bias.to(q.dtype))
        return self.o(y.transpose(1, 2).reshape(B, T, C)), present

class EffBlock(nn.Module):
    def __init__(self, d, h, kv, hidden, dropout, window):
        super().__init__()
        self.n1, self.n2 = RMSNorm(d), RMSNorm(d)
        self.attn, self.mlp = LocalGlobalAttention(d, h, kv, window), SwiGLU(d, hidden, dropout)
    def forward(self, x, bias, past=None, use_cache=False):
        a, present = self.attn(self.n1(x), bias, past, use_cache)
        x = x + a
        return x + self.mlp(self.n2(x)), present

class EfficientLM(nn.Module):
    """F: lightweight decoder. Sliding-window attention (window W) in most layers, full causal attention in every
    k-th layer, grouped-query attention (fewer key/value heads → smaller KV-cache), ALiBi instead of position
    embeddings. Local layers model nearby text; the few global layers can still reach any earlier token."""
    def __init__(self, vocab, n_layer, n_head, d_model, ctx_len, dropout=0.0, n_kv_head=4, ffn_hidden=2304,
                 window=256, global_every=4, **_):
        super().__init__()
        self.ctx_len = ctx_len
        self.wte = nn.Embedding(vocab, d_model)
        self.windows = [None if global_every and (i + 1) % global_every == 0 else window for i in range(n_layer)]
        self.blocks = nn.ModuleList([EffBlock(d_model, n_head, n_kv_head, ffn_hidden, dropout, w) for w in self.windows])
        self.register_buffer("slopes", torch.tensor(alibi_slopes(n_head)), persistent=False)
        self.norm_f = RMSNorm(d_model)
        self.lm_head = nn.Linear(d_model, vocab, bias=False); self.lm_head.weight = self.wte.weight
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.normal_(m.weight, 0.0, 0.02 / math.sqrt(2 * n_layer) if getattr(m, "RESIDUAL_SCALE", False) else 0.02)
            elif isinstance(m, nn.Embedding):
                nn.init.normal_(m.weight, 0.0, 0.02)
    def _bias(self, qpos, kpos, key_ok, window):
        """(B|1, H, Tq, Tk) additive mask: −slope·distance for allowed pairs, −inf elsewhere"""
        dist = (qpos[:, :, None] - kpos[:, None, :]).float()               # (B|1, Tq, Tk)
        ok = (dist >= 0) & key_ok[:, None, :]
        if window is not None: ok = ok & (dist < window)
        b = -self.slopes.view(1, -1, 1, 1) * dist[:, None]
        return b.masked_fill(~ok[:, None], float("-inf"))
    def forward(self, idx, targets=None, pos=None, mask=None, cache=None, use_cache=False, last_only=False):
        B, T = idx.shape
        if mask is None:                                          # training / plain forward: positions 0..T-1
            qpos = kpos = torch.arange(T, device=idx.device)[None]
            key_ok = torch.ones((1, T), dtype=torch.bool, device=idx.device)
        else:                                                     # generation: derive key positions from the mask
            key_ok = mask[:, 0, -1, :]                            # validity of every key (past + current)
            kpos = key_ok.long().cumsum(-1) - 1
            qpos = pos
        biases = {w: self._bias(qpos, kpos, key_ok, w) for w in set(self.windows)}
        x = self.wte(idx); new = []
        for i, blk in enumerate(self.blocks):
            x, c = blk(x, biases[self.windows[i]], cache[i] if cache is not None else None, use_cache); new.append(c)
        x = self.norm_f(x)
        if last_only: x = x[:, -1:]
        logits = self.lm_head(x)
        loss = None if targets is None else F.cross_entropy(logits.float().view(-1, logits.size(-1)), targets.reshape(-1))
        return logits, loss, (new if use_cache else None)

MODEL_REGISTRY = {"gpt2": GPT2LM, "modern": ModernLM, "encdec": EncDecLM, "retnet": RetNetLM, "hybrid": RetNetLM,
                  "prefixlm": PrefixLM, "efficient": EfficientLM}

def build_model(cfg, vocab):
    return MODEL_REGISTRY[cfg["arch"]](vocab=vocab, **cfg)

def count_params(m):
    return sum(p.numel() for p in m.parameters())      # tied weights counted once

In [5]:
# (backbone + adaptation methods copied unchanged from P3_02)
class Backbone(nn.Module):
    def __init__(self, lm, arch, attn):
        super().__init__()
        self.lm, self.arch = lm, arch
        self.bidir = attn == "bidir" or arch == "encdec"
        self.d = lm.wte.weight.shape[1]
        if arch == "encdec": lm.dec = nn.ModuleList()
        self.ctx_limit = lm.ctx_len if arch == "gpt2" else (2 * lm.ctx_len if arch in ("modern", "prefixlm", "encdec") else 10 ** 9)
    def blocks(self):
        return self.lm.enc if self.arch == "encdec" else self.lm.blocks
    def forward(self, idx, valid, prefix=None):
        lm, (B, T), dev = self.lm, idx.shape, idx.device
        P = 0 if prefix is None else prefix.size(0)
        x = lm.wte(idx)
        if P:
            x = torch.cat([prefix[None].expand(B, -1, -1).to(x.dtype), x], 1)
            valid = torch.cat([torch.ones(B, P, dtype=torch.bool, device=dev), valid], 1)
        L = T + P
        assert L <= self.ctx_limit, f"window {L} longer than the model's position range {self.ctx_limit}"
        pos = torch.arange(L, device=dev)[None]
        mask = valid[:, None, None, :] if self.bidir else None
        if self.arch == "encdec":
            cos, sin = lm._rope(0, L, dev)
            for blk in lm.enc: x = blk(x, cos, sin, mask)
            h = lm.enc_norm(x)
        elif self.arch == "gpt2":
            x = lm.drop(x + lm.wpe(pos))
            for blk in lm.blocks: x, _ = blk(x, None, mask, False)
            h = lm.ln_f(x)
        elif self.arch in ("modern", "prefixlm"):
            if lm.wpe is not None: x = x + lm.wpe(pos)
            cos, sin = lm.rope_cos[pos][:, None], lm.rope_sin[pos][:, None]
            x = lm.drop(x)
            for blk in lm.blocks: x, _ = blk(x, cos, sin, None, mask, False)
            h = lm.norm_f(x)
        elif self.arch == "efficient":
            dist = (pos[:, :, None] - pos[:, None, :]).float()
            biases = {}
            for w in set(lm.windows):
                if self.bidir:
                    ok = valid[:, None, :] & ((dist.abs() < w) if w is not None else torch.ones_like(dist, dtype=torch.bool))
                    ok = ok | torch.eye(L, dtype=torch.bool, device=dev)[None]
                    b = -lm.slopes.view(1, -1, 1, 1) * dist.abs()[:, None]
                else:
                    ok = dist >= 0
                    if w is not None: ok = ok & (dist < w)
                    b = -lm.slopes.view(1, -1, 1, 1) * dist[:, None]
                biases[w] = b.masked_fill(~ok[:, None], float("-inf"))
            for i, blk in enumerate(lm.blocks): x, _ = blk(x, biases[lm.windows[i]], None, False)
            h = lm.norm_f(x)
        else:
            raise ValueError(self.arch)
        return h[:, P:]

class LoRALinear(nn.Module):
    def __init__(self, base, r, alpha, rs=False):
        super().__init__()
        self.base, self.scale = base, alpha / (math.sqrt(r) if rs else r)
        self.A = nn.Parameter(torch.empty(r, base.in_features)); nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))
        self.B = nn.Parameter(torch.zeros(base.out_features, r))
    def forward(self, x):
        return self.base(x) + self.scale * F.linear(F.linear(x, self.A), self.B)

_PERM_IDX, _PERM_LOCK = {}, threading.Lock()
def perm_index(d, device):
    key = (d, str(device))
    with _PERM_LOCK:
        if key not in _PERM_IDX:
            i = torch.arange(d, device=device); _PERM_IDX[key] = (i[None, :] - i[:, None]) % d
        return _PERM_IDX[key]

class LoRMALinear(nn.Module):
    def __init__(self, base, r, alpha, variant="plus"):
        super().__init__()
        d = base.out_features; self.base, self.variant = base, variant
        if variant == "plus":
            self.scale = alpha / r
            self.A = nn.Parameter(torch.empty(r, d)); nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))
            self.B = nn.Parameter(torch.zeros(d, r))
        else:
            A = torch.zeros(r, d); A[0, 0] = 1.0
            B = torch.empty(d, r); nn.init.kaiming_uniform_(B, a=math.sqrt(5)); B[:, 0] = 1.0
            self.A, self.B = nn.Parameter(A), nn.Parameter(B)
    def forward(self, x):
        y = self.base(x)
        if self.variant == "plus":
            return y + self.scale * F.linear(F.linear(y, self.A), self.B)
        M = self.B @ self.A
        return F.linear(y, M.gather(1, perm_index(M.size(0), M.device)))

class IA3Linear(nn.Module):
    """(IA)^3: mode "out" rescales the layer's outputs (keys/values; `mask` limits it to part of a fused q|k|v
    output), mode "in" rescales its inputs (= the FFN hidden activation when wrapped around the down-projection)"""
    def __init__(self, base, mode, mask=None):
        super().__init__()
        self.base, self.mode = base, mode
        n = base.out_features if mode == "out" else base.in_features
        self.l = nn.Parameter(torch.ones(n))
        self.register_buffer("mask", mask if mask is not None else torch.ones(n, dtype=torch.bool), persistent=False)
    def forward(self, x):
        if self.mode == "in": return self.base(x * self.l.to(x.dtype))
        s = torch.where(self.mask, self.l, torch.ones_like(self.l))
        return self.base(x) * s.to(x.dtype)

class BiasLinear(nn.Module):
    def __init__(self, base):
        super().__init__(); self.base = base; self.bias = nn.Parameter(torch.zeros(base.out_features))
    def forward(self, x): return self.base(x) + self.bias

class AdapterLinear(nn.Module):
    def __init__(self, base, dim):
        super().__init__()
        d = base.out_features; self.base = base
        self.down, self.up = nn.Linear(d, dim), nn.Linear(dim, d)
        nn.init.normal_(self.down.weight, 0, 0.02); nn.init.zeros_(self.down.bias)
        nn.init.zeros_(self.up.weight); nn.init.zeros_(self.up.bias)
    def forward(self, x):
        y = self.base(x); return y + self.up(F.gelu(self.down(y)))

ATTN_TYPES = (CausalSelfAttention, ModernAttention, LocalGlobalAttention)
MLP_TYPES = (MLP, SwiGLU, GeluMLP)

def target_linears(backbone, which):
    out = []
    for parent in list(backbone.blocks().modules()):
        kind = "attn" if isinstance(parent, ATTN_TYPES) else "mlp" if isinstance(parent, MLP_TYPES) else None
        if kind is None or (which == "attn" and kind != "attn"): continue
        for name, child in list(parent.named_children()):
            if isinstance(child, nn.Linear):
                out.append((parent, name, child, bool(getattr(child, "RESIDUAL_SCALE", False))))
    return out

def apply_method(backbone, cfg):
    m = cfg["method"]
    for p in backbone.parameters(): p.requires_grad = (m == "full")
    if m in ("full", "frozen", "prompt"): return
    if m == "topk":
        for blk in list(backbone.blocks())[-cfg["topk"]:]:
            for p in blk.parameters(): p.requires_grad = True
        return
    if m == "ia3":
        for parent in list(backbone.blocks().modules()):
            if isinstance(parent, CausalSelfAttention):                      # fused q|k|v: rescale only k and v
                d = parent.qkv.in_features
                parent.qkv = IA3Linear(parent.qkv, "out", torch.arange(3 * d) >= d)
            elif isinstance(parent, (ModernAttention, LocalGlobalAttention)):
                parent.kvp = IA3Linear(parent.kvp, "out")
            elif isinstance(parent, SwiGLU): parent.w2 = IA3Linear(parent.w2, "in")
            elif isinstance(parent, (MLP, GeluMLP)): parent.proj = IA3Linear(parent.proj, "in")
        return
    which = cfg["lora_targets"] if m in ("lora", "rslora", "lorma") else "all"
    for parent, name, lin, is_out in target_linears(backbone, which):
        if m in ("lora", "rslora"): setattr(parent, name, LoRALinear(lin, cfg["lora_r"], cfg["lora_alpha"], rs=(m == "rslora")))
        elif m == "lorma": setattr(parent, name, LoRMALinear(lin, cfg["lora_r"], cfg["lora_alpha"], cfg["lorma_variant"]))
        elif m == "adapter":
            if is_out: setattr(parent, name, AdapterLinear(lin, cfg["adapter_dim"]))
        elif m == "bitfit":
            if lin.bias is not None: lin.bias.requires_grad = True
            else: setattr(parent, name, BiasLinear(lin))
    if m == "bitfit":
        for mod in backbone.modules():
            if isinstance(mod, (nn.LayerNorm, RMSNorm)):
                for p in mod.parameters(): p.requires_grad = True

def set_backbone_dropout(backbone, p):
    for mod in backbone.modules():
        if isinstance(mod, nn.Dropout): mod.p = p
        if isinstance(mod, (CausalSelfAttention, ModernAttention)): mod.dropout = p

_P1_CACHE, _P1_LOCK = {}, threading.Lock()
def load_phase1(key):
    with _P1_LOCK:
        if key not in _P1_CACHE:
            d = P1_DIRS[key]
            info = load_json(d / "run_info.json") or load_json(d / "summary.json") or {}
            pcfg = dict(info.get("cfg") or {}); assert pcfg, f"no run_info.json/summary.json next to {d}"
            sd = torch.load(d / "weights_final.pt", map_location="cpu", weights_only=False)
            _P1_CACHE[key] = (pcfg, {k: v.float() if v.is_floating_point() else v for k, v in sd.items()})
        return _P1_CACHE[key]

## 4. The retriever
`Backbone` (P3) runs the Phase-1 model without its LM head and returns the last hidden states; a chunk's vector is the
**masked mean** of its token states, L2-normalised; a case's vector (method `mean`) is the normalised mean of its
chunk vectors. Partial fine-tuning (`topk`) unfreezes the last k transformer blocks (B: of the encoder) and the final
norm. Smaller students (`keep_layers`) keep the first k blocks.

In [6]:
_P1_CACHE, _P1_LOCK = {}, threading.Lock()
def load_phase1(key):
    with _P1_LOCK:
        if key not in _P1_CACHE:
            d = P1_DIRS[key]; assert d is not None, f"Phase-1 weights for {key} not found"
            info = load_json(d / "run_info.json") or load_json(d / "summary.json") or {}
            pcfg = dict(info.get("cfg") or {}); assert pcfg
            sd = torch.load(d / "weights_final.pt", map_location="cpu", weights_only=False)
            _P1_CACHE[key] = (pcfg, {k: v.float() if v.is_floating_point() else v for k, v in sd.items()})
        return _P1_CACHE[key]

class Retriever(nn.Module):
    def __init__(self, bb):
        super().__init__(); self.bb = bb
    def forward(self, idx, valid):
        h = self.bb(idx, valid).float()
        m = valid.unsqueeze(-1).float()
        return F.normalize((h * m).sum(1) / m.sum(1).clamp(min=1.0), dim=-1)

def build_retriever(cfg, device):
    pcfg, sd = load_phase1(cfg["model"])
    torch.manual_seed(cfg["seed"])
    lm = MODEL_REGISTRY[pcfg["arch"]](vocab=sd["wte.weight"].shape[0], **{k: v for k, v in pcfg.items() if k not in ("vocab", "dropout")}, dropout=0.0)
    lm.load_state_dict(sd)
    k = cfg.get("keep_layers")
    if k:
        if pcfg["arch"] == "encdec": lm.enc = nn.ModuleList(list(lm.enc)[:k])
        else:
            lm.blocks = nn.ModuleList(list(lm.blocks)[:k])
            if pcfg["arch"] == "efficient": lm.windows = lm.windows[:k]
    bb = Backbone(lm, pcfg["arch"], cfg["attn"])
    set_backbone_dropout(bb, 0.0)                              # no dropout: gradient caching recomputes forwards exactly
    apply_method(bb, dict(cfg, method=cfg["method"]))
    if cfg["method"] == "topk":
        for n_ in ("ln_f", "norm_f", "enc_norm"):
            if getattr(lm, n_, None) is not None:
                for p in getattr(lm, n_).parameters(): p.requires_grad = True
    return Retriever(bb).to(device), pcfg

def count_parameters(model):
    seen, total, train = set(), 0, 0
    for n, p in model.named_parameters():
        if id(p) in seen: continue
        seen.add(id(p)); total += p.numel(); train += p.numel() if p.requires_grad else 0
    return total, train

def pad_batch(seqs, device):
    L = max(1, max(len(x) for x in seqs)); idx = np.full((len(seqs), L), PAD_ID, np.int64); v = np.zeros((len(seqs), L), bool)
    for i, x in enumerate(seqs): idx[i, :len(x)] = x; v[i, :len(x)] = True
    v[:, 0] = True                                             # an empty chunk still has one (pad) position
    return torch.from_numpy(idx).to(device), torch.from_numpy(v).to(device)

def doc_chunks(docs, d, cfg, cap):
    ch = docs.chunks(d, cfg["chunk_tok"])
    if cap and len(ch) > cap: ch = [ch[i] for i in np.linspace(0, len(ch) - 1, cap).round().astype(int)]
    return ch

def q_ctx_ranges(split, d, cfg, cap):
    ids = CTXS[split][d] if CTXS.get(split) is not None else []
    ch = DOCS[split]["q"].ctx_chunks(d, ids, cfg["chunk_tok"], cfg["ctx_window"], cap) if ids else []
    return ch or doc_chunks(DOCS[split]["q"], d, cfg, cap)

@torch.no_grad()
def encode_split(model, split, role, cfg, device, bs=48, ctx=False):
    """all chunks of all documents → (chunk vectors fp16 on device, owner document of every chunk);
    ctx=True: query citation-context windows instead of the evenly spread chunks"""
    docs = DOCS[split][role]; ranges, owner = [], []
    for d in range(len(docs)):
        ch = q_ctx_ranges(split, d, cfg, cfg["max_ctx"]) if ctx else doc_chunks(docs, d, cfg, cfg["max_chunks"])
        ranges += ch; owner += [d] * len(ch)
    order = np.argsort([b - a for a, b in ranges])[::-1]
    out = torch.empty(len(ranges), model.bb.d, dtype=torch.float16, device=device)
    amp = torch.autocast("cuda", dtype=torch.float16) if device.type == "cuda" else torch.autocast("cpu", enabled=False)
    model.eval()
    for i in range(0, len(order), bs):
        sel = order[i:i + bs]; idx, v = pad_batch([docs.ids(ranges[j]) for j in sel], device)
        with amp: out[torch.as_tensor(sel.copy(), device=device)] = model(idx, v).half()
    return out, torch.as_tensor(owner, device=device)

## 5. Scoring and metric
* **mean**: `cos(mean of query chunk vectors, mean of candidate chunk vectors)`.
* **MaxSim** (chunk-level late interaction, in the spirit of ColBERT (Khattab & Zaharia 2020) / BERT-PLI paragraph
  interactions (Shao et al. 2020)): for every query chunk the most similar chunk of the candidate, averaged over query
  chunks — a citation usually concerns one passage, which a whole-document average dilutes.
* **hybrid**: `z(dense) + λ·z(BM25)` with per-query z-normalisation; λ is chosen on dev. The BM25 scores are P7_01's
  best lexical system (n-gram BM25 fusion, possibly with the popularity prior).
* Micro-F1@7 exactly as the evaluator: `P = Σ hits / Σ 7`, `R = Σ hits / Σ |gold|` (gold not truncated).

In [7]:
def doc_means(vec, owner, n):
    s = torch.zeros(n, vec.size(1), device=vec.device).index_add_(0, owner, vec.float())
    c = torch.zeros(n, device=vec.device).index_add_(0, owner, torch.ones_like(owner, dtype=torch.float))
    return F.normalize(s / c.clamp(min=1)[:, None], dim=-1)

@torch.no_grad()
def score_matrices(qv, qo, cv, co, nq, nc, qcv=None, qco=None):
    qd, cd = doc_means(qv, qo, nq), doc_means(cv, co, nc)
    S_mean = (qd @ cd.T).float().cpu().numpy()
    S_max = np.zeros((nq, nc), np.float32); cvh = cv.half()
    for q in range(nq):
        Qc = qv[qo == q].half()
        sim = (Qc @ cvh.T).float()                                         # (query chunks, all candidate chunks)
        best = torch.full((Qc.size(0), nc), -2.0, device=sim.device).scatter_reduce(1, co.expand(Qc.size(0), -1), sim, reduce="amax", include_self=True)
        S_max[q] = best.mean(0).cpu().numpy()
    out = {"mean": S_mean, "maxsim": S_max, "both": z(S_mean) + z(S_max)}
    if qcv is not None:                                                    # citation-context query windows
        S_cm, S_cs = np.zeros((nq, nc), np.float32), np.zeros((nq, nc), np.float32); cdh = cd.half()
        for q in range(nq):
            Qc = qcv[qco == q].half()
            S_cm[q] = (Qc @ cdh.T).float().max(0).values.cpu().numpy()        # best context vs the case vector
            sim = (Qc @ cvh.T).float()
            best = torch.full((Qc.size(0), nc), -2.0, device=sim.device).scatter_reduce(1, co.expand(Qc.size(0), -1), sim, reduce="amax", include_self=True)
            S_cs[q] = best.mean(0).cpu().numpy()                             # contexts' best chunk, averaged
        out["ctx_max"], out["ctx_maxsim"] = S_cm, S_cs
    return out

def mask_self(S, s):
    S = S.copy()
    for qi, ci in SELF[s]: S[qi, ci] = -np.inf
    return S

def z(S):
    fin = np.isfinite(S); m = np.where(fin, S, np.nan)
    mu = np.nanmean(m, 1, keepdims=True); sd = np.nanstd(m, 1, keepdims=True) + 1e-6
    out = (S - mu) / sd; out[~fin] = -np.inf; return out

def topk(S, k):
    k = min(k, S.shape[1]); idx = np.argpartition(-S, k - 1, axis=1)[:, :k]
    o = np.argsort(-np.take_along_axis(S, idx, 1), axis=1)
    return np.take_along_axis(idx, o, 1)

def f1_at(S, s, K=7):
    order = topk(S, K); g = GOLD[s]["gold"]; ng = sum(GOLD[s]["n_gold"])
    hits = sum(len(set(row.tolist()) & set(gg)) for row, gg in zip(order, g))
    p = hits / (K * len(order)); r = hits / max(1, ng)
    return {"F1@7": 2 * p * r / max(1e-12, p + r), "P@7": p, "R@7": r}

LAMBDAS = (0.0, 0.25, 0.5, 1.0, 1.5, 2.0, 3.0, 5.0, 10.0)
def lex(s): return LEX[s]["best"].astype(np.float32)
def lexn(s): return LEX[s]["best_noyear"].astype(np.float32) if HAS_LEXPP else lex(s)
def late(s): return LEX[s]["late"].astype(np.float32)
def hybrid(S, s, lam, g=0.0):
    H = z(S) if lam == 0 else z(S) + lam * z(lexn(s))
    return H - g * late(s) if g else H

## 6. Training
**InfoNCE** (van den Oord et al. 2018; as in DPR, Karpukhin et al. 2020): a batch has `batch` training queries, each
with one randomly drawn cited case (positive) and `hard_negs` **BM25 hard negatives** (non-cited cases from BM25's
top `hard_from` for that query). For query `i` the loss is `−log softmax_j(cos(q_i, d_j)/τ)` at its positive, over
all cases of the batch (= in-batch negatives); other cases that are *also* cited by query `i` are removed from its
denominator (no false negatives). A case is represented by the mean of `chunks_train` random chunks (a new random
view each time — a cheap augmentation).

**Gradient caching** (Gao et al. 2021): a step holds ~70 chunks of 512 tokens — too many to back-propagate at once on
a T4. (1) embed all chunks without gradients, (2) compute the loss on the embeddings and its gradient w.r.t. every
chunk vector, (3) re-run each micro-batch of `micro` chunks with gradients and back-propagate that cached gradient.
Mathematically identical to one big batch (dropout is off); costs one extra forward pass.

**KD** (Hinton et al. 2015, applied to retrieval scores as in e.g. TAS-B, Hofstätter et al. 2021): the teacher's
cosine scores for every (training query, training case) pair are pre-computed by the teacher run
(`train_scores.npy`); the student matches the teacher's softmax distribution over the batch's cases:
`α·InfoNCE + (1−α)·T²·KL(softmax(t/τT) ‖ softmax(s/τT))`. No teacher forward pass during student training.

In [8]:
def mil_logits(E, owner_t, nq, ncand, cfg):
    """doc view: cos(mean query vector, mean case vector); ctx view: max over the query's context windows of
    cos(window, mean case vector) — multiple-instance: the case only has to match the passage that cites it"""
    if cfg["qview"] != "ctx":
        D = doc_means(E, owner_t, nq + ncand); return D[:nq] @ D[nq:].T
    qr = owner_t < nq
    cvv = doc_means(E[~qr], owner_t[~qr] - nq, ncand)
    sim = F.normalize(E[qr], dim=-1) @ cvv.T
    return torch.full((nq, ncand), -1e4, device=E.device, dtype=sim.dtype).scatter_reduce(0, owner_t[qr][:, None].expand(-1, ncand), sim, reduce="amax", include_self=False)

def train_queries():
    g = GOLD["train"]["gold"]; return [q for q in range(len(g)) if g[q]]

def hard_pool(q, cfg):
    top = LEX["train"]["top"][q][:cfg["hard_from"]]; gold = set(GOLD["train"]["gold"][q])
    selfc = {c for qi, c in SELF["train"] if qi == q}
    return [int(c) for c in top if int(c) not in gold and int(c) not in selfc]

def build_step(qs, rng, cfg):
    """→ list of documents (role, index) and positive slots; documents: queries first, then candidates"""
    cands, pos_slot, gold_sets = [], [], []
    for q in qs:
        g = GOLD["train"]["gold"][q]; p = int(rng.choice(g)); pos_slot.append(len(cands)); cands.append(p)
        hp = hard_pool(q, cfg)
        for c in (rng.choice(hp, min(cfg["hard_negs"], len(hp)), replace=False) if hp else []): cands.append(int(c))
        gold_sets.append(set(g))
    return cands, pos_slot, gold_sets

def chunk_views(docs, d, cfg, rng):
    ch = docs.chunks(d, cfg["chunk_tok"]); k = min(cfg["chunks_train"], len(ch))
    return [ch[i] for i in sorted(rng.choice(len(ch), k, replace=False))]

def train_model(cfg, device, log):
    model, pcfg = build_retriever(cfg, device); total, trainable = count_parameters(model)
    teacher_S = None
    if cfg["teacher"]:
        f = find_exp_file(cfg["teacher"], "train_scores.npy"); assert f is not None, f"train_scores.npy of {cfg['teacher']} not found — attach the run that trained it"
        teacher_S = np.load(f).astype(np.float32); log(f"teacher {cfg['teacher']}: train score matrix {teacher_S.shape}")
    params = [p for p in model.parameters() if p.requires_grad]
    qs_all = train_queries(); steps_ep = math.ceil(len(qs_all) / cfg["batch"]); total_upd = max(1, steps_ep * cfg["epochs"]); warm = max(1, int(cfg["warmup"] * total_upd))
    log(f"model {cfg['model']} ({pcfg['arch']}, {cfg['attn']}{', ' + str(cfg['keep_layers']) + ' layers' if cfg['keep_layers'] else ''}) method={cfg['method']} | "
        f"params {total/1e6:.1f}M, trainable {trainable/1e6:.2f}M ({100*trainable/total:.2f}%) | {len(qs_all)} training queries, {total_upd} updates")
    info = {"total_params": total, "trainable_params": trainable, "arch": pcfg["arch"], "history": [], "train_min": 0.0, "peak_mem_gb": None}
    if not params or cfg["epochs"] == 0: return model, info, set()
    opt = torch.optim.AdamW([{"params": [p for p in params if p.ndim >= 2], "weight_decay": cfg["weight_decay"]},
                             {"params": [p for p in params if p.ndim < 2], "weight_decay": 0.0}], lr=cfg["lr"], betas=(0.9, 0.98))
    use_amp = device.type == "cuda"; scaler = torch.amp.GradScaler("cuda", enabled=use_amp)
    amp = torch.autocast("cuda", dtype=torch.float16) if use_amp else torch.autocast("cpu", enabled=False)
    if use_amp: torch.cuda.reset_peak_memory_stats(device)
    rng = np.random.default_rng(cfg["seed"]); upd, t0 = 0, time.time()
    Dq, Dc = DOCS["train"]["q"], DOCS["train"]["c"]
    for ep in range(1, cfg["epochs"] + 1):
        model.train(); order = rng.permutation(qs_all); run, nb = 0.0, 0
        for i in range(0, len(order), cfg["batch"]):
            qs = [int(x) for x in order[i:i + cfg["batch"]]]
            cands, pos_slot, gold_sets = build_step(qs, rng, cfg)
            seqs, owner = [], []
            for j, q in enumerate(qs):
                if cfg["qview"] == "ctx":
                    rs = q_ctx_ranges("train", q, cfg, None)
                    if len(rs) > cfg["ctx_train"]: rs = [rs[k] for k in sorted(rng.choice(len(rs), cfg["ctx_train"], replace=False))]
                else: rs = chunk_views(Dq, q, cfg, rng)
                for r in rs: seqs.append(Dq.ids(r)); owner.append(j)
            for j, c in enumerate(cands):
                for r in chunk_views(Dc, c, cfg, rng): seqs.append(Dc.ids(r)); owner.append(len(qs) + j)
            owner_t = torch.as_tensor(owner, device=device); nd = len(qs) + len(cands)
            mbs = [list(range(a, min(a + cfg["micro"], len(seqs)))) for a in range(0, len(seqs), cfg["micro"])]
            with torch.no_grad():                                           # (1) embeddings without graph
                embs = []
                for mb in mbs:
                    idx, v = pad_batch([seqs[k] for k in mb], device)
                    with amp: embs.append(model(idx, v).float())
            E = torch.cat(embs).detach().requires_grad_(True)               # (2) loss on embeddings
            logits = mil_logits(E, owner_t, len(qs), len(cands), cfg) / cfg["tau"]
            mask = torch.zeros_like(logits, dtype=torch.bool)
            for a, gs in enumerate(gold_sets):
                for b_, c in enumerate(cands):
                    if c in gs and b_ != pos_slot[a]: mask[a, b_] = True
            target = torch.as_tensor(pos_slot, device=device)
            loss = F.cross_entropy(logits.masked_fill(mask, -1e4), target)
            if teacher_S is not None:
                T = cfg["kd_T"]; t = torch.as_tensor(teacher_S[np.ix_(qs, cands)], device=device) / cfg["tau"]
                kd = F.kl_div(F.log_softmax(logits / T, -1), F.log_softmax(t / T, -1), log_target=True, reduction="batchmean") * T * T
                loss = cfg["kd_alpha"] * loss + (1 - cfg["kd_alpha"]) * kd
            loss.backward(); G = E.grad.detach()
            for mb in mbs:                                                  # (3) re-forward with graph, inject cached gradient
                idx, v = pad_batch([seqs[k] for k in mb], device)
                with amp: e = model(idx, v).float()
                scaler.scale(e).backward(gradient=G[mb[0]:mb[-1] + 1])
            lr_mult = (upd + 1) / warm if upd < warm else max(0.0, (total_upd - upd) / max(1, total_upd - warm))
            for g_ in opt.param_groups: g_["lr"] = cfg["lr"] * lr_mult
            scaler.unscale_(opt); torch.nn.utils.clip_grad_norm_(params, cfg["grad_clip"])
            scaler.step(opt); scaler.update(); opt.zero_grad(set_to_none=True); upd += 1
            run += loss.item(); nb += 1
            if nb % 25 == 0: log(f"  epoch {ep} step {nb}/{steps_ep}: loss {run / nb:.4f} ({(time.time() - t0)/60:.1f} min)")
        msg = {"epoch": ep, "train_loss": run / max(nb, 1), "min": (time.time() - t0) / 60}
        info["history"].append(msg); log(json.dumps({k: round(v, 4) if isinstance(v, float) else v for k, v in msg.items()}))
    info.update(train_min=(time.time() - t0) / 60, peak_mem_gb=torch.cuda.max_memory_allocated(device) / 1e9 if use_amp else None)
    return model, info, {n for n, p in model.named_parameters() if p.requires_grad}

def save_trainable(model, names, path):
    sd = {n: q.detach().half().cpu() if q.is_floating_point() else q.detach().cpu() for n, q in model.named_parameters() if n in names}
    torch.save(sd, str(path) + ".tmp"); os.replace(str(path) + ".tmp", path)

def load_trained(name, device):
    cp, w = find_exp_file(name, "config.json"), find_exp_file(name, "trainable.pt")
    assert cp is not None and w is not None, f"p7/exps/{name}/(config.json, trainable.pt) not found — attach the run that trained {name}"
    tcfg = dict(DEFAULTS); tcfg.update(load_json(cp))
    m, _ = build_retriever(tcfg, device)
    sd = torch.load(w, map_location="cpu", weights_only=False)
    res = m.load_state_dict({k: v.float() if v.is_floating_point() else v for k, v in sd.items()}, strict=False)
    assert not res.unexpected_keys, res.unexpected_keys[:5]
    return m, tcfg

## 7. Experiment runner
Dev: both scoring variants alone and in the hybrid for every λ → the best (variant, λ) on dev is used for test.
Writes `submission_dense.zip` (best dense variant, no BM25) and `submission_hybrid.zip` (best variant + λ·BM25), plus
the dev/test score matrices (`scores.npz`) for later ensembles. The teacher run also writes `train_scores.npy`.

In [9]:
def write_submission(S, path):
    top = topk(S, 7); cids, qids = IDS["test"]["candidates"], IDS["test"]["queries"]
    pred = {}
    for qi, q in enumerate(qids):
        ids = [cids[j] for j in top[qi]]
        assert len(ids) == len(set(ids)) == min(7, len(cids)) and canon(q) not in {canon(x) for x in ids}
        pred[q] = ids
    assert len(pred) == len(qids)
    with zipfile.ZipFile(path, "w", zipfile.ZIP_DEFLATED) as zf: zf.writestr("prediction.json", json.dumps(pred, indent=1))

def run_experiment(name, device):
    cfg = exp_cfg(name); d = EXPS_DIR / name; d.mkdir(parents=True, exist_ok=True)
    if (d / "result.json").exists(): print(f"[{name}] already done — skipped"); return
    logf = open(d / "log.txt", "a")
    def log(msg):
        line = f"[{name} | {device}] {msg}"; print(line, flush=True); logf.write(line + "\n"); logf.flush()
    save_json(cfg, d / "config.json")
    if cfg["from_exp"]:                                          # re-score a finished experiment (no training)
        model, scfg = load_trained(cfg["from_exp"], device)
        src = load_json(find_exp_file(cfg["from_exp"], "result.json")); info = dict(src["info"], train_min=0.0)
        cfg = dict(scfg, **{k: v for k, v in EXPERIMENTS[name].items()}, name=name, save_train_scores=False)
        log(f"re-scoring {cfg['from_exp']} (no training)"); names = set()
    else:
        model, info, names = train_model(cfg, device, log)
        if names: save_trainable(model, names, d / "trainable.pt")
    t0 = time.time()
    qv, qo = encode_split(model, "dev", "q", cfg, device); cv, co = encode_split(model, "dev", "c", cfg, device)
    enc_s = time.time() - t0; n_chunks = len(qo) + len(co)
    qcv, qco = encode_split(model, "dev", "q", cfg, device, ctx=True) if CTXS["dev"] is not None else (None, None)
    Sd = {k: mask_self(v, "dev") for k, v in score_matrices(qv, qo, cv, co, len(DOCS["dev"]["q"]), len(DOCS["dev"]["c"]), qcv, qco).items()}
    res, best = {}, None
    for var, S in Sd.items():
        res[f"dense {var}"] = f1_at(S, "dev")
        for lam in LAMBDAS:
            for g in YEAR_GAMMAS:
                r = f1_at(hybrid(S, "dev", lam, g), "dev"); res[f"hybrid {var} λ={lam} γ={g}"] = r
                if best is None or r["F1@7"] > best[3]: best = (var, lam, g, r["F1@7"])
    res["BM25 alone"] = f1_at(lex("dev"), "dev")
    best_dense = max(Sd, key=lambda v: res[f"dense {v}"]["F1@7"])
    log("dev: " + " | ".join(f"{k} {v['F1@7']:.4f}" for k, v in res.items() if k.startswith("dense") or k == "BM25 alone")
        + f" | best hybrid {best[0]} λ={best[1]} γ={best[2]} {best[3]:.4f} | encoding {n_chunks} chunks in {enc_s:.0f}s")
    save_scores = {f"dev_{k}": v.astype(np.float16) for k, v in Sd.items()}
    test_min = None
    if cfg["test"]:
        t0 = time.time()
        qv, qo = encode_split(model, "test", "q", cfg, device); cv, co = encode_split(model, "test", "c", cfg, device)
        qcv, qco = encode_split(model, "test", "q", cfg, device, ctx=True) if CTXS["test"] is not None else (None, None)
        St = {k: mask_self(v, "test") for k, v in score_matrices(qv, qo, cv, co, len(DOCS["test"]["q"]), len(DOCS["test"]["c"]), qcv, qco).items()}
        write_submission(St[best_dense], d / "submission_dense.zip")
        write_submission(hybrid(St[best[0]], "test", best[1], best[2]), d / "submission_hybrid.zip")
        save_scores.update({f"test_{k}": v.astype(np.float16) for k, v in St.items()}); test_min = (time.time() - t0) / 60
    np.savez_compressed(d / "scores.npz", **save_scores)
    if cfg["save_train_scores"]:
        t0 = time.time()
        cv, co = encode_split(model, "train", "c", cfg, device); nq_, nc_ = len(DOCS["train"]["q"]), len(DOCS["train"]["c"])
        if cfg["qview"] == "ctx":                                   # same score as in training: best context vs case vector
            qv, qo = encode_split(model, "train", "q", cfg, device, ctx=True); cd_ = doc_means(cv, co, nc_).half()
            Str = mask_self(np.stack([(qv[qo == q].half() @ cd_.T).float().max(0).values.cpu().numpy() for q in range(nq_)]), "train")
        else:
            qv, qo = encode_split(model, "train", "q", cfg, device)
            Str = mask_self((doc_means(qv, qo, nq_) @ doc_means(cv, co, nc_).T).float().cpu().numpy(), "train")
        Str[~np.isfinite(Str)] = -1.0
        np.save(d / "train_scores.npy", Str.astype(np.float16)); log(f"saved train_scores.npy {Str.shape} in {(time.time()-t0)/60:.1f} min")
    row = {"name": name, "model": cfg["model"] + (f"[{cfg['keep_layers']}L]" if cfg.get("keep_layers") else ""), "attn": cfg["attn"],
           "method": cfg["method"] + (f"+KD({cfg['teacher']})" if cfg["teacher"] else "") + (" ctx-trained" if cfg["qview"] == "ctx" else "") + (f" [re-scored {cfg['from_exp']}]" if cfg["from_exp"] else ""),
           "dev F1@7 dense mean": res["dense mean"]["F1@7"], "dev F1@7 dense maxsim": res["dense maxsim"]["F1@7"],
           "dev F1@7 dense ctx_max": res.get("dense ctx_max", {}).get("F1@7"), "dev F1@7 dense ctx_maxsim": res.get("dense ctx_maxsim", {}).get("F1@7"),
           "dev F1@7 hybrid": best[3], "hybrid": f"{best[0]} λ={best[1]} γ={best[2]}", "dev P@7 hybrid": res[f"hybrid {best[0]} λ={best[1]} γ={best[2]}"]["P@7"],
           "dev R@7 hybrid": res[f"hybrid {best[0]} λ={best[1]} γ={best[2]}"]["R@7"], "dev F1@7 BM25": res["BM25 alone"]["F1@7"],
           "total_params_M": info["total_params"] / 1e6, "trainable_%": 100 * info["trainable_params"] / info["total_params"],
           "peak_mem_GB": info["peak_mem_gb"], "train_min": info["train_min"], "encode_ms_per_chunk": 1000 * enc_s / max(1, n_chunks), "test_min": test_min}
    save_json({"row": row, "info": info, "cfg": cfg, "dev": res}, d / "result.json")
    log(f"DONE dense {res[f'dense {best_dense}']['F1@7']:.4f} ({best_dense}) | hybrid {best[3]:.4f} | BM25 {res['BM25 alone']['F1@7']:.4f}")
    logf.close(); del model
    if device.type == "cuda": torch.cuda.empty_cache()

## 8. Unit tests (CPU, seconds)
* chunking keeps every token exactly once and respects the length limit;
* gradient caching gives the same gradients as one ordinary backward pass;
* MaxSim and Micro-F1@7 against brute force.

In [10]:
def _tests():
    torch.manual_seed(0)
    dq = DOCS["dev"]["c"]
    for dd in range(min(5, len(dq))):
        ch = dq.chunks(dd, 64); a0, b0 = int(dq.so[dq.ds[dd]]), int(dq.so[dq.ds[dd + 1]])
        assert ch[0][0] == a0 and ch[-1][1] == b0 and all(b - a <= 64 for a, b in ch) and all(ch[i][1] == ch[i + 1][0] for i in range(len(ch) - 1))
    lin = nn.Linear(6, 4)
    class Toy(nn.Module):
        def __init__(s): super().__init__(); s.l = lin
        def forward(s, x): return F.normalize(s.l(x), dim=-1)
    toy = Toy(); X = torch.randn(10, 6); owner = torch.tensor([0, 0, 1, 1, 1, 2, 3, 3, 4, 5])
    def lossf(E):
        D = doc_means(E, owner, 6); return F.cross_entropy(D[:3] @ D[3:].T / 0.05, torch.tensor([0, 1, 2]))
    lossf(toy(X)).backward(); g_direct = lin.weight.grad.clone(); lin.weight.grad = None; lin.bias.grad = None
    with torch.no_grad(): E = toy(X)
    E = E.requires_grad_(True); lossf(E).backward(); G = E.grad
    for a in (0, 4, 8): toy(X[a:a + 4]).backward(gradient=G[a:a + 4])
    assert torch.allclose(g_direct, lin.weight.grad, atol=1e-5), "gradient caching mismatch"
    qv = F.normalize(torch.randn(5, 8), dim=-1); qo = torch.tensor([0, 0, 1, 1, 1]); cv = F.normalize(torch.randn(7, 8), dim=-1); co = torch.tensor([0, 1, 1, 2, 2, 2, 0])
    S = score_matrices(qv, qo, cv, co, 2, 3)["maxsim"]
    for q in range(2):
        for c in range(3):
            ref = (qv[qo == q] @ cv[co == c].T).max(1).values.mean().item(); assert abs(S[q, c] - ref) < 2e-3, (S[q, c], ref)
    S = np.random.rand(len(GOLD["dev"]["gold"]), len(DOCS["dev"]["c"])).astype(np.float32)
    for q, g in enumerate(GOLD["dev"]["gold"]): S[q, g] += 10
    r = f1_at(S, "dev"); ng = np.array(GOLD["dev"]["n_gold"]); hits = sum(min(7, len(g)) for g in GOLD["dev"]["gold"])
    assert abs(r["P@7"] - hits / (7 * len(ng))) < 1e-9 and abs(r["R@7"] - hits / ng.sum()) < 1e-9
    E = torch.randn(7, 4, requires_grad=True); own = torch.tensor([0, 0, 1, 2, 2, 3, 3])
    lg = mil_logits(E, own, 2, 2, dict(DEFAULTS, qview="ctx")); lg.sum().backward()
    Ec = F.normalize(E.detach(), dim=-1); cvv = doc_means(E.detach()[3:], own[3:] - 2, 2)
    ref = torch.stack([torch.stack([(Ec[own == q] @ cvv[c]).max() for c in range(2)]) for q in range(2)])
    assert torch.allclose(lg.detach(), ref, atol=1e-5) and E.grad is not None and E.grad.abs().sum() > 0
    print("unit tests passed")
_tests()

unit tests passed


## 9. Run the queue (one worker thread per GPU; a KD student waits for its teacher if both are in the queue)

In [11]:
def run_queue(names):
    def dep_of(n): return exp_cfg(n)["teacher"]
    missing = [f"{n} needs {dep_of(n)}" for n in names if dep_of(n) and dep_of(n) not in names
               and not (EXPS_DIR / n / "result.json").exists() and find_exp_file(dep_of(n), "train_scores.npy") is None]
    missing += [f"{n} needs {exp_cfg(n)['from_exp']}" for n in names if exp_cfg(n)["from_exp"] and not (EXPS_DIR / n / "result.json").exists()
                and find_exp_file(exp_cfg(n)["from_exp"], "trainable.pt") is None]
    if missing: print("WARNING — these will fail, attach the runs that trained:\n  " + "\n  ".join(missing))
    pending, running, failed, lock = list(names), set(), set(), threading.Lock()
    def take():
        with lock:
            for n in pending:
                dep = dep_of(n)
                if dep in failed: pending.remove(n); failed.add(n); print(f"[{n}] not run: {dep} failed"); return "retry"
                if dep and (dep in pending or dep in running): continue
                pending.remove(n); running.add(n); return n
            return "wait" if pending else None
    devices = [torch.device(f"cuda:{i}") for i in range(NUM_GPUS)] or [torch.device("cpu")]
    def worker(dev):
        while True:
            name = take()
            if name is None: return
            if name in ("wait", "retry"):
                if name == "wait": time.sleep(20)
                continue
            ok = False
            if not (EXPS_DIR / name / "result.json").exists() and time_left_h() < exp_cfg(name)["est_h"] + 0.2:
                print(f"[{name}] skipped: not enough session time left")
            else:
                try:
                    if dev.type == "cuda":
                        with torch.cuda.device(dev): run_experiment(name, dev)
                    else: run_experiment(name, dev)
                    ok = (EXPS_DIR / name / "result.json").exists()
                except Exception:
                    print(f"[{name}] FAILED:\n{traceback.format_exc()}", flush=True)
                    (EXPS_DIR / name).mkdir(parents=True, exist_ok=True); (EXPS_DIR / name / "error.txt").write_text(traceback.format_exc())
            with lock:
                running.discard(name)
                if not ok: failed.add(name)
    threads = [threading.Thread(target=worker, args=(dv,)) for dv in devices]
    for t in threads: t.start()
    for t in threads: t.join()

t_q = time.time(); run_queue(RUN_QUEUE); print(f"queue finished in {(time.time() - t_q)/60:.1f} min")

[E_lorma_ctx | cuda:0] model E (prefixlm, bidir) method=lorma | params 113.2M, trainable 3.05M (2.69%) | 817 training queries, 824 updates
[E_lorma_rs | cuda:1] re-scoring E_lorma (no training)
[E_lorma_rs | cuda:1] dev: dense mean 0.3635 | dense maxsim 0.3680 | dense both 0.3724 | dense ctx_max 0.3902 | dense ctx_maxsim 0.3902 | BM25 alone 0.4758 | best hybrid ctx_maxsim λ=0.5 γ=3.0 0.4836 | encoding 12815 chunks in 134s
[E_lorma_ctx | cuda:0]   epoch 1 step 25/103: loss 2.2478 (2.6 min)
[E_lorma_ctx | cuda:0]   epoch 1 step 50/103: loss 2.1275 (5.1 min)
[E_lorma_rs | cuda:1] DONE dense 0.3902 (ctx_max) | hybrid 0.4836 | BM25 0.4758
[A0_6L_kd_rs | cuda:1] re-scoring A0_6L_kd (no training)
[A0_6L_kd_rs | cuda:1] dev: dense mean 0.3302 | dense maxsim 0.3391 | dense both 0.3469 | dense ctx_max 0.3613 | dense ctx_maxsim 0.3380 | BM25 alone 0.4758 | best hybrid mean λ=1.0 γ=100.0 0.4814 | encoding 12815 chunks in 37s
[E_lorma_ctx | cuda:0]   epoch 1 step 75/103: loss 2.0322 (7.7 min)
[A0_6

## 10. Results

In [12]:
rows = [load_json(f)["row"] for f in sorted(EXPS_DIR.glob("*/result.json"))]
if rows:
    df = pd.DataFrame(rows).sort_values("dev F1@7 hybrid", ascending=False)
    display(df.round(4)); df.to_csv(ROOT / "p7" / "p7_experiments.csv", index=False)
    print("BM25 alone on dev (P7_01 best lexical system):", round(f1_at(lex("dev"), "dev")["F1@7"], 4))
for f in sorted(EXPS_DIR.glob("*/error.txt")): print(f"FAILED: {f.parent.name}\n{f.read_text()[-1500:]}")

,name,model,attn,method,dev F1@7 dense mean,dev F1@7 dense maxsim,dev F1@7 hybrid,hybrid,dev P@7 hybrid,dev R@7 hybrid,dev F1@7 BM25,total_params_M,trainable_%,peak_mem_GB,train_min,encode_ms_per_chunk,test_min,dev F1@7 dense ctx_max,dev F1@7 dense ctx_maxsim
9,E_lorma_r2,E,bidir,lorma,0.3546,0.3613,0.4836,ctx_maxsim λ=1.5 γ=100.0,0.5266,0.4471,0.4758,113.1671,2.6929,2.8529,74.7218,10.1770,3.9669,0.3969,0.3758
2,A0_6L_kd_ctx,A0[6L],causal,full+KD(E_lorma_ctx) ctx-trained,0.3369,0.3357,0.4836,ctx_max λ=1.0 γ=100.0,0.5266,0.4471,0.4758,68.4810,100.0000,1.9728,25.9182,2.7159,1.0602,0.3713,0.3535
10,E_lorma_rs,E,bidir,lorma [re-scored E_lorma],0.3635,0.3680,0.4836,ctx_maxsim λ=0.5 γ=3.0,0.5266,0.4471,0.4758,113.1671,2.6929,2.8578,0.0000,10.4308,4.1748,0.3902,0.3902
3,A0_6L_kd_rs,A0[6L],causal,full+KD(E_lorma) [re-scored A0_6L_kd],0.3302,0.3391,0.4814,mean λ=1.0 γ=100.0,0.5242,0.4450,0.4758,68.4810,100.0000,1.9705,0.0000,2.8912,1.0962,0.3613,0.3380
8,E_lorma_ctx,E,bidir,lorma ctx-trained,0.3469,0.3535,0.4803,mean λ=5.0 γ=100.0,0.5230,0.4440,0.4758,113.1671,2.6929,2.8543,81.2351,9.9088,3.8492,0.3902,0.3791
6,E_full,E,bidir,full,0.3546,0.3669,0.4380,mean λ=0.5,0.4770,0.4049,0.4158,110.1197,100.0000,4.1981,20.5579,7.9087,2.8774,NaN,NaN
7,E_lorma,E,bidir,lorma,0.3635,0.3680,0.4358,maxsim λ=0.25,0.4746,0.4029,0.4158,113.1671,2.6929,2.8578,22.2162,9.5151,3.4721,NaN,NaN
4,A1_full,A1,causal,full,0.3569,0.3680,0.4358,mean λ=0.5,0.4746,0.4029,0.4158,110.1197,100.0000,4.1986,19.8412,7.5573,2.7568,NaN,NaN
1,A0_6L_kd,A0[6L],causal,full+KD(E_lorma),0.3302,0.3380,0.4291,mean λ=0.5,0.4673,0.3967,0.4158,68.4810,100.0000,1.9705,6.9218,2.5932,0.9485,NaN,NaN
0,A0_6L_control,A0[6L],causal,full,0.3346,0.3335,0.4291,mean λ=0.5,0.4673,0.3967,0.4158,68.4810,100.0000,1.9684,7.0209,2.6234,0.9587,NaN,NaN


BM25 alone on dev (P7_01 best lexical system): 0.4758
